# HTBoost **pooled** v5 — one cross-sectional model over the full multi-country swap universe

**This is the pooled counterpart to `model_htboost_v5.ipynb`.** It fits **ONE** HTBoost model on
the **full pool of ALL securities** (the frozen 88-security multi-country universe), predicts on
**ALL** securities, and emits rows that are **byte-for-byte mergeable** with v5's
`v5_metrics_long.csv`. It supersedes the abandoned `model_htboost_v3` pilot. Where v3 and v5 differ,
**v5 wins**; where this notebook and v5 differ on *anything in the shared schema*, they must not.

**Goal: the best _honest_ out-of-sample result** — best OOS, never best in-sample, never by selecting
against test folds.

### What "pooled" changes vs v5 (per-security)
- **ONE model per `(horizon × fold)`** over the stacked panel of every security, with
  `security` / `country` / `maturity` as **categorical** features so the single model can exploit
  cross-sectional and maturity structure. **No per-security training loops.**
- **Predict on ALL securities.** Each `(horizon × fold × sample)` emits (a) an **aggregate pooled**
  row over the whole panel and (b) a **per-security** row for **every** security in the universe.
  The NOR subset aligns 1:1 with v5.
- **Value-of-categoricals (WI4).** The pooled model is fit **with** and **without** the
  `security/country/maturity` categoricals; the OOS delta is reported (do the identifiers add skill?).

### What is held identical to v5 (the comparability contract)
1. **Metrics code.** Every row is produced by the *same* `compute_metrics_row` — imported from the
   shared `htb_metrics.py` (a verbatim extraction of v5's metrics cell). `SHARED_COLS` is asserted
   equal to the v5 list. RW(=0) benchmark, Campbell–Thompson OOS-R², Clark–West, Diebold–Mariano–
   Harvey (HAC lag = H−1), Pesaran–Timmermann, one-sided binomial; **directional p-values use
   `n_eff` at h>1**.
2. **Target(s) & horizons.** Outright level change `y = level.shift(-h) − level`
   (`target_kind='level_change'`); identical `H_GRID = [1,5,21,63]` (+ gated `[126,252]`); the
   excess-return target at the long end exactly as v5 switches. **No** v3 residual / relative-value /
   RV-slope / butterfly / cross-country families in the primary path.
3. **Axes.** Identical `WF_FOLDS` (GFC/ZIRP/COVID/Hiking regime walk-forward), `BLOCK_CV_FOLDS`,
   `EMBARGO_FOR_H`, `validation_scheme ∈ {walk_forward, block_cv, loro}`, and the **identical PCA
   rule** (xm_* → ≤`XM_PCA_KMAX` comps at ≥`XM_PCA_VAR` train variance, fit on **train rows only**).
4. **Schema, pooled tag.** Shared columns emitted with `model_kind='pooled'`, `is_pooled=True`;
   pooled-only diagnostics are **additional** columns — shared ones are never renamed or dropped.
5. **Honest tuning.** `modality="accurate"` does HTBoost's internal block-CV tuning. We do **NOT**
   call `HTBcv`, do **NOT** grid or force `lambda`. `FROZEN_CONFIG` records only
   `loss / modality / nfold / priortype`; HTBoost tunes depth/trees/regularisation itself.
6. **Universe frozen on training data only** (reuse the v3 freeze: `universe_freeze.json`, `min_obs`,
   88 securities). Never re-derived from any test period.
7. **Determinism.** A pinned Julia seed and a cached feature panel make reported numbers reproducible.
8. **Carry stays export-only (Part C).** `carry_roll` may remain a point-in-time *feature* for parity
   with v5; the target is never carry. No v3-style long/short Sharpe is presented as model skill.

### Decision rule (stated before running)
Pooled confirms the null if aggregate OOS DirAcc is noise-consistent and no `(horizon × fold)` cell
survives multiple-testing correction. A genuine lead requires MTC survival **and** positive DirAcc
across **multiple** regimes (not Hiking-only). Block-CV is a **non-causal** learnability diagnostic
read under the same ex-ante asymmetric rule as v5 — never a forecast, never in Part C.

## Setup
Import the scientific-Python stack, the statistical-test libraries, the Julia bridge (`juliacall`) used to call HTBoost, the project's `data_loader` helper, and the shared `htb_metrics` module — a verbatim extraction of v5's metrics cell — so every results row is produced by v5's exact metrics code.

In [ ]:
import re
import os
import sys
import json
import hashlib
import warnings
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest, binom, norm
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from juliacall import Main as jl

# Put the repo root (dir containing src/) on sys.path so the flat root-level module(s)
# (thesis_style, htb_metrics) import whether cwd is the repo root or notebooks/.
_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)
from src.data.bloomberg import load_data  # was: data_loader shim (broke from notebooks/ cwd)
import src.config as config  # shared constants (JULIA_SEED, XM_PCA_*)
# Shared thesis figure style (Okabe-Ito palette, vector PDF, despined axes, regime colours).
from thesis_style import apply_thesis_style, OKABE_ITO, REGIME_COLORS, SERIES_COLOR
apply_thesis_style()

# Shared metrics contract — the SAME code v5 uses (verbatim extraction).
import htb_metrics
from htb_metrics import (compute_metrics_row, clark_west, dm_harvey,
                         pesaran_timmermann, config_hash, SHARED_COLS, _score)

warnings.filterwarnings('ignore', category=FutureWarning)
print('Imports OK (pooled v5: shared htb_metrics + PCA/StandardScaler, statsmodels HAC, scipy.norm)')

### Julia runtime
Load the Julia packages HTBoost needs and remove any leftover worker processes so each fit runs single-process, then pin the RNG seed so `modality='accurate'`'s internal cross-validation is reproducible.

In [ ]:
# ── Julia runtime setup (same invariants as v4/v5) + pinned seed (determinism) ─
jl.seval('using DataFrames')
jl.seval('using HybridTreeBoosting')
jl.seval('using Distributed')
jl.seval('using SharedArrays')
jl.seval('using Dates')
jl.seval('using Random')
jl.seval(
    'ws = workers(); '
    'if length(ws) > 0 && ws != [1]; rmprocs(ws); end'
)
# Pin the RNG once here; the bridge re-seeds before every HTBfit so the whole sweep
# is reproducible across runs (constraint #6).
jl.seval(f'Random.seed!({config.JULIA_SEED})')  # from config (single source); cell runs before CONFIG
print(f'Julia procs after cleanup: {jl.nprocs()}  (should be 1)')
print('Julia RNG seeded with 20260619 (re-seeded before every fit in the bridge).')

### Configuration
All run parameters in one place: horizons, the walk-forward and block-CV fold dates, the embargo, the PCA rule, the minimum-observation thresholds, the cross-sectional categorical identifiers, and the switches that gate the heavy sweeps. The only model-config difference from v5 is deliberate — lambda is never forced or gridded; `modality='accurate'` tunes depth/trees/regularisation by internal block CV. None of these are chosen from out-of-sample results.

In [ ]:
# ── CONFIG — change only here ────────────────────────────────────────────────
# Pooled v5 keeps EVERY axis identical to v5 (horizon grid, WF folds, block-CV folds,
# embargo, PCA rule, target). The ONLY model-config difference: we do NOT force/grid
# lambda — modality='accurate' tunes depth/trees/regularisation by internal block CV.
# NOTHING here is chosen by looking at an OOS/regime fold.

# ── Primary horizon + the pre-committed horizon GRID (WI5) — identical to v5 ──
H                = 5            # default/primary horizon (smoke, leakage audit, Part C default)
H_GRID           = [1, 5, 21, 63]          # 1d, 1w, 1m, 3m — always run
H_GRID_LONG      = [126, 252]              # 6m, 12m — only if data length supports the folds
OVERLAP          = H - 1                    # label-overlap purge; ALWAYS H-1 per horizon (invariant)

LOSS             = 't'
MODALITY         = 'accurate'  # strongest HTBoost setting; does internal block-CV tuning for OOS
PRIORTYPE        = 'hybrid'    # HTBoost prior; part of FROZEN_CONFIG (no other hyperparam forced)
NFOLD_INTERNAL   = 4           # block CV (randomizecv=False); default 'accurate' value
UNIVERSE_MIN_OBS = 500         # reused from the v3 freeze (documented, not re-derived here)
MIN_TRAIN_OBS    = 252         # ≥1 year training per fold (pooled: per-fold panel rows)
MIN_TEST_OBS     = 20          # ≥20 OOS obs to score a fold (aggregate panel)
MIN_TEST_OBS_SEC = 5           # ≥5 OOS obs to score a per-security breakdown row (else NaN row)
ALPHA_MT         = 0.05        # multiple-testing alpha
SMOKE_SEC        = 'NOR_10Y'   # the NOR pilot tenor used for smoke prints / horizon gating
TARGET_LAGS      = 5           # AR lags of chg_1d; fixed, identical for all securities/horizons

OUT_DIR = f'{_ROOT}/results/pooled_v5'   # repo-root-anchored (cwd-independent), so results/ resolves under <root> whether Jupyter starts at root or notebooks/; auto-push only stages files under <root>/results/
os.makedirs(OUT_DIR, exist_ok=True)

# Determinism (constraint #6): pin the Julia RNG so 'accurate' internal CV is reproducible.
JULIA_SEED = config.JULIA_SEED

# ── WI1: PCA compression of the cross-market xm_* block — IDENTICAL RULE to v5 ─
# The xm_* block is the named overfitting surface. In the pooled panel it is larger
# (~2×|universe| cols). We keep the v5 rule UNCHANGED so the PCA step is identical
# across both notebooks (the comparability contract, point 3). PCA is fit on the
# fold's TRAINING rows only, applied to train+test, and lives in the FIT PATH.
#   k = smallest #components explaining ≥ XM_PCA_VAR of TRAIN variance, capped at XM_PCA_KMAX.
XM_PCA_ENABLE    = True
XM_PCA_VAR       = 0.95        # explained-variance target (a-priori) — same as v5
XM_PCA_KMAX      = 12          # hard cap on #components (a-priori) — SAME as v5 (identical PCA rule)

# ── WI5: NO forced/gridded lambda (pooled prompt §HTBoost-config) ────────────
# modality='accurate' performs automatic hyperparameter tuning by internal block CV and
# optimises for OOS; the author discourages user-side CV/tuning on top of it. So we do
# NOT call HTBcv and do NOT set lambda/depth/ntrees — HTBoost tunes them. FROZEN_CONFIG
# records ONLY loss/modality/nfold/priortype and is reused EVERYWHERE.
FROZEN_CONFIG = dict(loss=LOSS, modality=MODALITY, nfold=NFOLD_INTERNAL, priortype=PRIORTYPE)

# ── WI4: categorical (cross-sectional) identifiers ───────────────────────────
# Passed to HTBoost as STRING columns (auto-detected categorical; cleaned separately,
# not filled 0.0). The value-of-categoricals cell fits with and without these.
CATEGORICAL_COLS = ['security', 'country', 'maturity']

# ── WI6: block-CV / leave-one-regime-out folds + embargo — identical to v5 ───
BLOCK_CV_FOLDS = [
    ('Block_GFC',    '2007-01-01', '2012-12-31', 'GFC'),
    ('Block_ZIRP',   '2013-01-01', '2019-12-31', 'ZIRP'),
    ('Block_COVID',  '2020-01-01', '2021-12-31', 'COVID'),
    ('Block_Hiking', '2022-01-01', '2026-12-31', 'Hiking'),
]
def EMBARGO_FOR_H(h):
    """López de Prado embargo (business days); scales with H. Two-sided in block-CV."""
    return max(int(h), 10)

# ── Run switches (gate the heavy sweeps) ─────────────────────────────────────
RUN_WALK_FORWARD = True
RUN_BLOCK_CV     = True
RUN_LORO         = True
RUN_NO_CATS      = False       # WI4 (no-cats ablation) intentionally disabled for the single committed run
# QUICK_MODE is a DEVELOPMENT/EXECUTION-BUDGET switch only — it does NOT change any
# model-selection decision (those stay frozen). It shrinks modality / grid / universe so
# the whole notebook runs end-to-end quickly to verify wiring. Default OFF.
QUICK_MODE       = False
QUICK_MODALITY   = 'fast'
QUICK_HGRID      = [5]
QUICK_UNIVERSE   = ['NOR_1Y', 'NOR_5Y', 'NOR_10Y', 'EUR_5Y', 'EUR_10Y',
                    'SWE_5Y', 'SWE_10Y', 'SOFR_5Y', 'SOFR_10Y']  # small multi-country subset

# ── Comparability: write where v5's comparison cell reads (POOLED_CSV_PATH) ───
# v5's c29 comparison cell reads 'results/v5_nor/pooled_metrics_long.csv'.
V5_OUT_DIR       = f'{_ROOT}/results/v5_nor'
POOLED_CSV_PATH  = f'{V5_OUT_DIR}/pooled_metrics_long.csv'   # the path v5 merges automatically
LOCAL_LONG_CSV   = f'{OUT_DIR}/pooled_metrics_long.csv'      # local copy too

# ── Walk-forward folds: expanding window, test windows map to regimes — v5 ───
WF_FOLDS = [
    # (name,          test_start,    test_end,      regime)
    ('GFC',           '2010-01-01',  '2012-12-31',  'GFC'),
    ('ZIRP_early',    '2013-01-01',  '2016-12-31',  'ZIRP'),
    ('ZIRP_late',     '2017-01-01',  '2019-12-31',  'ZIRP'),
    ('COVID',         '2020-01-01',  '2021-12-31',  'COVID'),
    ('Hiking',        '2022-01-01',  '2026-12-31',  'Hiking'),
]
FOLD_NAMES = [f[0] for f in WF_FOLDS]

NOTEBOOK_TAG = 'pooled_v5'
RUN_TS = pd.Timestamp.utcnow().isoformat()
# Thread provenance tags into the shared metrics module (used by compute_metrics_row).
htb_metrics.NOTEBOOK_TAG = NOTEBOOK_TAG
htb_metrics.RUN_TS = RUN_TS

# Feature spec string folded into config_hash so it tracks the PCA/target settings.
FEAT_SPEC   = f'pca_var{XM_PCA_VAR}_kmax{XM_PCA_KMAX}_tl{TARGET_LAGS}_pooled'
FROZEN_HASH = config_hash(FROZEN_CONFIG, extra=FEAT_SPEC)

print(f'H(primary)={H}  H_GRID={H_GRID}  (+long {H_GRID_LONG} if data supports)')
print(f'LOSS={LOSS!r} MODALITY={MODALITY!r} PRIORTYPE={PRIORTYPE!r} NFOLD_INTERNAL={NFOLD_INTERNAL}')
print(f'FROZEN_CONFIG (no forced lambda/depth/ntrees) = {FROZEN_CONFIG}')
print(f'config_hash = {FROZEN_HASH}')
print(f'PCA(xm_*): enable={XM_PCA_ENABLE} var≥{XM_PCA_VAR} kmax={XM_PCA_KMAX}  (IDENTICAL rule to v5)')
print(f'categoricals: {CATEGORICAL_COLS}   value-of-categoricals (no-cats) run: {RUN_NO_CATS}')
print(f'switches: WF={RUN_WALK_FORWARD} BLOCK_CV={RUN_BLOCK_CV} LORO={RUN_LORO} QUICK_MODE={QUICK_MODE}')
print(f'OUT_DIR={OUT_DIR!r}   POOLED_CSV_PATH={POOLED_CSV_PATH!r}')
print(f'Walk-forward folds ({len(WF_FOLDS)}):')
for f in WF_FOLDS:
    print(f'  {f[0]:<14s}  test [{f[1]} → {f[2]}]  regime={f[3]}')
print(f'Block-CV folds ({len(BLOCK_CV_FOLDS)}):')
for f in BLOCK_CV_FOLDS:
    print(f'  {f[0]:<14s}  block [{f[1]} → {f[2]}]  regime={f[3]}')

### Load data and define the universe
Load the swap panel, identify the swap columns, guard against duration columns leaking in, and freeze the multi-country study universe by reusing the v3 freeze (88 securities, `min_obs`). Membership is derived from training data only and never re-derived from a test period.

In [ ]:
# ── Data load + FROZEN multi-country universe (reuse the v3 freeze) ──────────
# Constraint #4: the universe is frozen on TRAINING data only and never re-derived
# from a test period. We REUSE the v3 freeze (htboost_results_v3/universe_freeze.json:
# 88 securities, multi-country, min_obs=500, cutoff 2022-01-01). If that file is
# missing we re-freeze from PRE-FIRST-FOLD training data with the SAME min_obs and
# write the record — never using any WF test-window obs for membership.
df_raw = load_data()

_SWAP_PAT = re.compile(r'^[A-Z]+_\d+[WMY]$')
SWAP_COLS = sorted([c for c in df_raw.columns if _SWAP_PAT.match(c)])

# Duration.xlsx is NOT loaded by load_data() — guard anyway (no DV01 leakage)
dur_cols = [c for c in df_raw.columns if 'uration' in c.lower() or 'DV01' in c]
assert len(dur_cols) == 0, f'Duration columns found — exclude: {dur_cols}'

print(f'df_raw shape:    {df_raw.shape}')
print(f'Date range:      {df_raw.index.min().date()} → {df_raw.index.max().date()}')
print(f'Swap columns:    {len(SWAP_COLS)}')

_V3_FREEZE = 'htboost_results_v3/universe_freeze.json'
_FREEZE_OUT = f'{OUT_DIR}/universe_freeze.json'


def _refreeze(cutoff_date):
    """Re-derive the universe from pre-cutoff training data only (no look-ahead)."""
    train = df_raw[df_raw.index < pd.Timestamp(cutoff_date)][SWAP_COLS]
    obs = train.notna().sum()
    uni = sorted(obs[obs >= UNIVERSE_MIN_OBS].index.tolist())
    rec = {'cutoff_date': cutoff_date, 'min_obs': UNIVERSE_MIN_OBS,
           'universe': uni, 'obs_counts': {s: int(obs[s]) for s in uni},
           'source': 'refrozen_pre_first_fold'}
    return uni, rec


if os.path.exists(_V3_FREEZE):
    with open(_V3_FREEZE) as _f:
        _freeze = json.load(_f)
    UNIVERSE = sorted(_freeze['universe'])
    _freeze['source'] = 'reused_v3_universe_freeze'
    print(f'\nReused v3 freeze: {len(UNIVERSE)} securities '
          f'(min_obs={_freeze.get("min_obs")}, cutoff={_freeze.get("cutoff_date")})')
else:
    # Freeze on data strictly BEFORE the first walk-forward test window.
    _cut = WF_FOLDS[0][1]
    UNIVERSE, _freeze = _refreeze(_cut)
    print(f'\n[no v3 freeze] re-froze {len(UNIVERSE)} securities on pre-{_cut} training data.')

# Keep only securities actually present in the current data load (defensive).
UNIVERSE = [s for s in UNIVERSE if s in df_raw.columns]
# Persist the freeze record used by THIS run.
with open(_FREEZE_OUT, 'w') as _f:
    json.dump({**_freeze, 'universe': UNIVERSE,
               'n': len(UNIVERSE)}, _f, indent=2)

_countries = sorted({s.rsplit('_', 1)[0] for s in UNIVERSE})
print(f'Frozen universe written to {_FREEZE_OUT}')
print(f'Universe: {len(UNIVERSE)} securities across {len(_countries)} countries: {_countries}')
assert len(UNIVERSE) >= 5, 'frozen universe too small — check data load / freeze file'
assert SMOKE_SEC in UNIVERSE, f'{SMOKE_SEC} not in frozen universe'
print(f'NOR subset (aligns 1:1 with v5): '
      f'{[s for s in UNIVERSE if s.startswith("NOR_")]}')

# ── Decision 2: pooled PREDICTION/SCORING target = the Norwegian tenors only ──
# The pooled arm still TRAINS on the full frozen universe (cross-sectional pooling is
# the whole point); we only PREDICT/SCORE the Norwegian securities, which is what the
# results section compares against the per-security v5 model. This removes most of the
# per-security scoring cost and shrinks the output. Training scope is NOT subset.
SCORE_UNIVERSE = [s for s in UNIVERSE if s.startswith('NOR_')]
assert SCORE_UNIVERSE, 'no NOR_ securities in frozen universe — pooled scoring target is empty'
print(f'SCORE_UNIVERSE (prediction/scoring target — NOR tenors): {SCORE_UNIVERSE}')
print(f'  TRAINING panel = full universe ({len(UNIVERSE)} secs);  '
      f'SCORING restricted to {len(SCORE_UNIVERSE)} NOR tenors.')

### Macro/global feature builder
Define the timing-alignment policy and `add_macro_features_v4`, which precomputes the global macro, volatility, equity, commodity, credit and IBOR features. The policy lag keeps every exogenous feature observable at decision time, identical to v5.

In [ ]:
# ── TIMING ALIGNMENT POLICY (v4 vs v3) ──────────────────────────────────────
# v3 used same-day VIX/MOVE/equity/commodity values. This is safe for US
# securities (SOFR) where the feature and fixing are both US-close. For non-US
# fixings (EUR, NOR fix intraday before NY close), same-day US-close data is
# unavailable at decision time. v4 conservatively shifts ALL US-close exogenous
# series by 1 day: feature[t] = series[t-1], guaranteed observable at any market.
#
# Series NOT shifted (contemporaneous OK):
#   - own-security swap rate: target and feature are the same daily close
#   - country IBOR rates: same-country same-session fix
#   - monthly macro: already ffill+shift(1) in v3; unchanged
#
# Series shifted by 1 (v4 change from v3):
#   - VIX, MOVE, V2X, VXN, RVX, OVX, GVZ
#   - SPX, SX5E, MXWO, NDX
#   - Oil, copper, OPEC, natgas
#   - Breakeven inflation (TIPS), IG/HY credit spreads
#   - Additional: CPURNSA, PCE CORE
#
# Cross-market swap rates: shift(1) applied in _features_for_security_v4.

MATURITY_YEARS = {
    '1W': 1/52, '1M': 1/12, '3M': 0.25, '6M': 0.5,
    '1Y': 1.0,  '5Y': 5.0,  '10Y': 10.0, '15Y': 15.0, '30Y': 30.0,
}
COUNTRY_PRIMARY_IBOR = {
    'NOR':   ('NIBOR3M Index  (L1)',  'NIBOR6M Index'),
    'SWE':   ('STIB3M Index  (L1)',   None),
    'EUR':   ('EUR006M Index  (L1)',  'EUR003M Index'),
    'SOFR':  ('SOFRRATE Index  (L1)', None),
    'CAN':   ('CAONREPO Index  (L1)', None),
    'AUS':   ('BBSW3M Index  (L1)',   None),
    'POL':   ('WIBR3M Index  (L1)',   None),
    'BRAZ':  ('BZDIOVRA Index  (R2)', None),
    'CHIN':  ('CNRR007 Index  (R2)',  None),
    'TURK':  ('MUTKCALM Index  (R1)', None),
    'SONIA': ('SONIO/N Index  (L1)',  None),
}


def add_macro_features_v4(df):
    df = df.copy()

    def _safe(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=df.index)

    def _s1(col):
        return _safe(col).shift(1)

    def _mom_s1(col, w, mp):
        return _safe(col).pct_change().rolling(w, min_periods=mp).sum().shift(1)

    def _diff_roll_s1(col, w, mp):
        return _safe(col).diff(1).rolling(w, min_periods=mp).sum().shift(1)

    # MOVE — shift(1): US rates vol, only final after US close
    move = _safe('MOVE Index  (R3)')
    df['move_level']  = move.shift(1)
    df['move_chg_1d'] = move.diff().shift(1)
    mv_mu = move.rolling(252, min_periods=60).mean()
    mv_sd = move.rolling(252, min_periods=60).std()
    df['move_zscore'] = ((move - mv_mu) / (mv_sd + 1e-9)).clip(-5, 5).shift(1)

    # VIX — shift(1): US 4pm close
    vix = _safe('^VIX')
    df['vix_level']  = vix.shift(1)
    vx_mu = vix.rolling(252, min_periods=60).mean()
    vx_sd = vix.rolling(252, min_periods=60).std()
    df['vix_zscore'] = ((vix - vx_mu) / (vx_sd + 1e-9)).clip(-5, 5).shift(1)

    # Additional vol indices — shift(1)
    for src, dst in [('VXN Index  (R2)', 'vxn_level'),
                     ('RVX Index  (L2)', 'rvx_level'),
                     ('OVX Index  (R3)', 'ovx_level'),
                     ('GVZ Index  (R2)', 'gvz_level'),
                     ('V2X Index  (L2)', 'v2x_level')]:
        df[dst] = _s1(src)

    # Equity — shift(1)
    for src, lbl in [('SPX Index  (R4)',  'spx'),
                     ('SX5E Index  (R4)', 'sx5e'),
                     ('MXWO Index  (R4)', 'mxwo'),
                     ('NDX Index  (L4)',  'ndx')]:
        df[f'{lbl}_mom_1m'] = _mom_s1(src, 21, 10)
        df[f'{lbl}_mom_3m'] = _mom_s1(src, 63, 30)

    # Commodities — shift(1)
    df['oil_mom_1m']    = _mom_s1('CL1 COMB Comdty  (R3)', 21, 10)
    df['oil_mom_3m']    = _mom_s1('CL1 COMB Comdty  (R3)', 63, 30)
    df['copper_mom_1m'] = _mom_s1('C01 Comdty  (R4)', 21, 10)
    df['opec_prod']     = _s1('OPECDALY Index  (R3)')
    df['natgas_mom_1m'] = _mom_s1('MUC1 Comdty  (L2)', 21, 10)

    # Breakeven inflation — shift(1): TIPS market (US)
    for bc in ['breakeven_5Y', 'breakeven_10Y']:
        be = _safe(bc)
        df[f'{bc}_level']  = be.shift(1)
        df[f'{bc}_chg_1m'] = be.diff(21).shift(1)
    df['breakeven_slope'] = (_safe('breakeven_10Y') - _safe('breakeven_5Y')).shift(1)

    # Credit spreads — shift(1): US credit market
    df['ig_spread'] = _s1('IG_spread')
    df['hy_spread'] = _s1('HY_spread')
    df['ig_chg_1m'] = _safe('IG_spread').diff(21).shift(1)
    df['hy_chg_1m'] = _safe('HY_spread').diff(21).shift(1)

    # Extra inflation — shift(1)
    df['cpi_nsa_level']  = _s1('CPURNSA Index  (L3)')
    df['pce_core_level'] = _s1('PCE CORE Index  (L2)')

    # Monthly macro — ffill+shift(1): same logic as v3
    for src, dst in [
        ('CPI YOY Index  (R1)',  'cpi_yoy'),
        ('PCE CRCH Index  (L1)', 'pce_core_chg'),
        ('NFP TCH Index  (L4)',  'nfp_change'),
        ('NAPMPMI Index  (R2)',  'pmi_us'),
        ('ECCPEMUY Index  (R1)', 'pmi_eur'),
        ('UMRTEMU Index  (R1)',  'unemp_eur'),
    ]:
        df[dst] = _safe(src).ffill().shift(1)

    # Country IBOR rates — contemporaneous (own-country daily fix)
    for country, (col3m, col6m) in COUNTRY_PRIMARY_IBOR.items():
        sr3m = _safe(col3m)
        df[f'{country}_ibor_level']      = sr3m
        df[f'{country}_ibor_chg_1d']     = sr3m.diff()
        df[f'{country}_ibor_mom_1m']     = sr3m.diff().rolling(21, min_periods=10).sum()
        sr6m = _safe(col6m) if col6m else pd.Series(np.nan, index=df.index)
        df[f'{country}_ibor_term_slope'] = sr6m - sr3m

    # Additional EURIBOR + NIBOR tenors — contemporaneous
    for src, dst in [
        ('EUR003M Index', 'eur_3m'),
        ('EUR001W Index', 'eur_1w'),
        ('NIBOR1M Index', 'nor_ibor_1m'),
        ('NIBOR6M Index', 'nor_ibor_6m'),
    ]:
        df[dst] = _safe(src)

    return df


print('add_macro_features_v4 defined')

### Curve helpers
Small helpers for the swap annuity, linear curve interpolation, and the carry/roll spread used in the per-security feature set and the Part C export.

In [ ]:
def swap_annuity(S_pct, T_years):
    if pd.isna(S_pct) or S_pct <= 0.01:
        return np.nan
    S_semi = S_pct / 200
    n = 2 * T_years
    return (1 - (1 + S_semi) ** (-n)) / (S_pct / 100)


def interp_rate(country, T_target, df):
    pts = {}
    for m, T in MATURITY_YEARS.items():
        col = f'{country}_{m}'
        if col in df.columns:
            pts[T] = df[col]
    if not pts:
        return pd.Series(np.nan, index=df.index)
    ts = sorted(pts.keys())
    if T_target <= 0 or T_target <= ts[0]:
        return pd.Series(0.0, index=df.index)
    if T_target >= ts[-1]:
        return pts[ts[-1]]
    for i in range(len(ts) - 1):
        if ts[i] <= T_target <= ts[i + 1]:
            w = (T_target - ts[i]) / (ts[i + 1] - ts[i])
            return pts[ts[i]] * (1 - w) + pts[ts[i + 1]] * w
    return pd.Series(np.nan, index=df.index)


def carry_roll_spread(sec, df, country, maturity):
    T = MATURITY_YEARS.get(maturity, np.nan)
    if np.isnan(T):
        return pd.Series(np.nan, index=df.index)
    S_n   = df[sec]
    T_roll = max(0.0, T - 1.0)
    S_nm1  = (pd.Series(0.0, index=df.index) if T_roll == 0.0
              else interp_rate(country, T_roll, df))
    return S_n - S_nm1


print('Helper functions defined: swap_annuity, interp_rate, carry_roll_spread')

### Per-security feature set and pooled panel builder
Assemble the uniform per-security feature matrix — own-curve momentum/vol/level features, contemporaneous IBOR, the lagged global block, and a frozen cross-market `xm_*` block — then `build_panel_pooled` stacks every security into one long panel and attaches the h-day-ahead target. Pooled differences vs v5: the Norway-specific block is disabled so the feature set is uniform across countries, the `xm_*` layout is constant for every security, and `security`/`country`/`maturity` ride along as categoricals carrying the cross-sectional identity.

**Design note (pooled arm — country-uniform feature set).** The pooled arm deliberately uses a *country-uniform* feature set: the Norway-specific drivers (Norges-Bank/SSB policy rates, EUR-NOK/USD-NOK, the I-44 index and `swap_govt_spread`) are **excluded** here and remain available only to the per-security arm. A single pooled model requires one uniform column layout across all 88 securities; features defined for only one country would be missing for the other 87 (or force ragged columns) and break the pooled fit. Cross-sectional identity is carried by the `security`/`country`/`maturity` categoricals, not by country-specific columns. The contrast between the two arms — uniform-pooled vs Norway-rich per-security — is itself part of the comparison.

In [ ]:
# ── WI2: pooled feature function = v5's feature set MINUS the Norway block ────
# This reuses v5's `_features_for_security_v4` logic VERBATIM for every global /
# cross-market / curve / momentum / vol / macro / credit / carry_roll feature, and:
#   • DISABLES the country=='NOR' Norges-Bank/SSB/EUR-NOK/USD-NOK/I-44 block and
#     `swap_govt_spread` (Norway-only) → the feature set is UNIFORM across the
#     multi-country panel (pooled prompt WI2 + "Do NOT include Norway-specific
#     features or any feature defined for only some countries").
#   • Makes the cross-market `xm_*` block UNIFORM: it iterates a FROZEN list of the
#     universe swaps (XM_SWAPS) for EVERY security (including its own lagged level —
#     backward-looking, leak-free), so every security contributes the SAME xm columns
#     and the stacked panel has no ragged union. The single pooled model then sees a
#     constant column layout; cross-sectional identity is carried by the
#     security/country/maturity categoricals, not by which xm columns exist.
# All xm_* / exogenous series are shift(1); own-security features are contemporaneous
# (same daily close). This passes the leakage audit unchanged.

# Frozen cross-market swap set: the universe swaps, in a fixed order (determinism).
XM_SWAPS = sorted([s for s in UNIVERSE if _SWAP_PAT.match(s)])


def _xm_block(df):
    """Uniform cross-market block: xm_<swap>_lv = swap[t-1], xm_<swap>_ch = Δswap[t-1].
    Same columns for every security (computed once, reused across the panel)."""
    out = {}
    for col in XM_SWAPS:
        if col not in df.columns:
            continue
        fn = 'xm_' + re.sub(r'[^a-zA-Z0-9]', '_', col).strip('_')
        out[fn + '_lv'] = df[col].shift(1)
        out[fn + '_ch'] = df[col].diff(1).shift(1)
    return pd.DataFrame(out, index=df.index)


def _features_for_security_pooled(sec, df, country, maturity, xm_block=None):
    """v5 feature set (global + curve + momentum + vol + macro + credit + carry_roll
    + uniform cross-market xm_*) for one security. NO Norway-specific block."""

    def _get(col):
        return df[col].copy() if col in df.columns else pd.Series(np.nan, index=df.index)

    level  = df[sec]
    chg_1d = level.diff(1)

    # Explicit autoregressive target lags (backward-looking: lag-k at t uses chg_1d[t-k]).
    _tlags = {f'chg_lag{k}': chg_1d.shift(k) for k in range(1, TARGET_LAGS + 1)}

    mom_1m   = chg_1d.rolling(21,  min_periods=10).sum()
    mom_3m   = chg_1d.rolling(63,  min_periods=30).sum()
    mom_6m   = chg_1d.rolling(126, min_periods=60).sum()
    mom_12m  = chg_1d.rolling(252, min_periods=120).sum()
    mom_63d  = chg_1d.rolling(63,  min_periods=21).mean()
    dev_ma_3m  = level - level.rolling(63,  min_periods=21).mean()
    dev_ma_12m = level - level.rolling(252, min_periods=120).mean()
    rvol_20d   = chg_1d.rolling(20, min_periods=5).std()
    vol_mu     = rvol_20d.rolling(252, min_periods=60).mean()
    vol_sd     = rvol_20d.rolling(252, min_periods=60).std()
    vol_med    = rvol_20d.rolling(252, min_periods=60).median()
    vol_regime = (rvol_20d > vol_med).astype(float)
    vol_zscore = ((rvol_20d - vol_mu) / (vol_sd + 1e-9)).clip(-5, 5)
    lv_mu      = level.rolling(504, min_periods=120).mean()
    lv_sd      = level.rolling(504, min_periods=120).std()
    lv_zscore  = ((level - lv_mu) / (lv_sd + 1e-9)).clip(-5, 5)
    carry_roll = carry_roll_spread(sec, df, country, maturity)

    def _slope(lm, sm):
        lc, sc = f'{country}_{lm}', f'{country}_{sm}'
        if lc in df.columns and sc in df.columns:
            return df[lc] - df[sc]
        return pd.Series(np.nan, index=df.index)

    sofr_mom_1m = (df['SOFR_10Y'].diff(1).rolling(21, min_periods=10).sum()
                   if 'SOFR_10Y' in df.columns
                   else pd.Series(np.nan, index=df.index))

    base = {
        **_tlags,
        'carry_roll'         : carry_roll,
        'mom_63d'            : mom_63d,
        'mom_1m'             : mom_1m,
        'mom_3m'             : mom_3m,
        'mom_6m'             : mom_6m,
        'mom_12m'            : mom_12m,
        'dev_ma_3m'          : dev_ma_3m,
        'dev_ma_12m'         : dev_ma_12m,
        'vol_20d'            : rvol_20d,
        'vol_regime'         : vol_regime,
        'vol_zscore'         : vol_zscore,
        'level_zscore'       : lv_zscore,
        'slope_10_1'         : _slope('10Y', '1Y'),
        'slope_10_5'         : _slope('10Y', '5Y'),
        'slope_5_1'          : _slope('5Y',  '1Y'),
        'sofr_mom_1m'        : sofr_mom_1m,
        # country IBOR (contemporaneous, same-session fix)
        'ibor_level'         : _get(f'{country}_ibor_level'),
        'ibor_chg_1d'        : _get(f'{country}_ibor_chg_1d'),
        'ibor_mom_1m'        : _get(f'{country}_ibor_mom_1m'),
        'ibor_term_slope'    : _get(f'{country}_ibor_term_slope'),
        'swap_ibor_basis'    : level - _get(f'{country}_ibor_level'),
        # global vol (shift(1) baked into precomputed columns)
        'move_level'         : _get('move_level'),
        'move_zscore'        : _get('move_zscore'),
        'vix_level'          : _get('vix_level'),
        'vix_zscore'         : _get('vix_zscore'),
        'v2x_level'          : _get('v2x_level'),
        'vxn_level'          : _get('vxn_level'),
        'ovx_level'          : _get('ovx_level'),
        'gvz_level'          : _get('gvz_level'),
        # equity (shift(1))
        'spx_mom_1m'         : _get('spx_mom_1m'),
        'spx_mom_3m'         : _get('spx_mom_3m'),
        'sx5e_mom_1m'        : _get('sx5e_mom_1m'),
        'sx5e_mom_3m'        : _get('sx5e_mom_3m'),
        'mxwo_mom_1m'        : _get('mxwo_mom_1m'),
        'ndx_mom_1m'         : _get('ndx_mom_1m'),
        # commodities (shift(1))
        'oil_mom_1m'         : _get('oil_mom_1m'),
        'oil_mom_3m'         : _get('oil_mom_3m'),
        'copper_mom_1m'      : _get('copper_mom_1m'),
        'opec_prod'          : _get('opec_prod'),
        'natgas_mom_1m'      : _get('natgas_mom_1m'),
        # breakeven + credit (shift(1))
        'be5y_level'         : _get('breakeven_5Y_level'),
        'be10y_level'        : _get('breakeven_10Y_level'),
        'be_slope'           : _get('breakeven_slope'),
        'be5y_chg_1m'        : _get('breakeven_5Y_chg_1m'),
        'ig_spread'          : _get('ig_spread'),
        'hy_spread'          : _get('hy_spread'),
        'ig_chg_1m'          : _get('ig_chg_1m'),
        'hy_chg_1m'          : _get('hy_chg_1m'),
        # monthly macro (ffill+shift(1))
        'cpi_yoy'            : _get('cpi_yoy'),
        'pce_core_chg'       : _get('pce_core_chg'),
        'nfp_change'         : _get('nfp_change'),
        'pmi_us'             : _get('pmi_us'),
        'pmi_eur'            : _get('pmi_eur'),
        'unemp_eur'          : _get('unemp_eur'),
        'cpi_nsa_level'      : _get('cpi_nsa_level'),
        'pce_core_level'     : _get('pce_core_level'),
        # extra IBOR tenors (contemporaneous) — global columns, NaN→0 where absent
        'eur_3m'             : _get('eur_3m'),
        'eur_1w'             : _get('eur_1w'),
        'nor_ibor_1m'        : _get('nor_ibor_1m'),
        'nor_ibor_6m'        : _get('nor_ibor_6m'),
    }

    xm = xm_block if xm_block is not None else _xm_block(df)
    base_df = pd.DataFrame(base, index=df.index)
    # No Norway-specific block (disabled for the uniform multi-country panel).
    return pd.concat([base_df, xm], axis=1)


# ── Pooled panel builder: stack ALL securities, attach categoricals ──────────
META_COLS = {'date', 'security', 'country', 'maturity', 'y', 'level'}


def build_panel_pooled(df_raw, securities, h, xm_block=None):
    """Stacked multi-security panel. Target: outright h-day change y = level[t+h]−level[t].
    Adds `country`/`maturity`/`security` string columns (categoricals). The uniform xm
    block is computed once and reused across securities for speed + a constant layout."""
    if xm_block is None:
        xm_block = _xm_block(df_raw)
    rows = []
    for sec in securities:
        if sec not in df_raw.columns:
            continue
        country, maturity = sec.rsplit('_', 1)
        level = df_raw[sec]
        feats = _features_for_security_pooled(sec, df_raw, country, maturity, xm_block=xm_block)
        y     = level.shift(-h) - level
        sec_df = feats.reset_index()
        if sec_df.columns[0] != 'date':
            sec_df = sec_df.rename(columns={sec_df.columns[0]: 'date'})
        sec_df['security'] = sec
        sec_df['country']  = country
        sec_df['maturity'] = maturity
        sec_df['y']        = y.values
        sec_df['level']    = level.values
        rows.append(sec_df)
    if not rows:
        return pd.DataFrame()
    panel = pd.concat(rows, ignore_index=True)
    panel = panel.dropna(subset=['y'])
    panel = panel.sort_values(['date', 'security']).reset_index(drop=True)
    return panel


# Numeric feature columns (everything not meta and not a categorical string col).
def feature_columns(panel, use_categoricals=True):
    numeric = [c for c in panel.columns if c not in META_COLS and c not in CATEGORICAL_COLS]
    cats = list(CATEGORICAL_COLS) if use_categoricals else []
    return numeric, cats


print('Pooled feature builder defined: _features_for_security_pooled, _xm_block, '
      'build_panel_pooled, feature_columns')
print(f'  cross-market XM_SWAPS frozen at {len(XM_SWAPS)} swaps '
      f'→ {2*len(XM_SWAPS)} xm_* columns (uniform across securities)')
print(f'  Norway-specific block DISABLED (uniform multi-country feature set)')

### Apply the macro/global features
Run the macro feature builder on the raw panel, report the derived columns, and freeze the shared cross-market `XM_BLOCK` once so `build_panel_pooled` reuses an identical layout for every security and horizon. The builder derives columns from raw source series, so re-running recomputes identical values rather than double-applying.

In [ ]:
# ── Augment df_raw with all macro/global features, then freeze the xm block ──
# add_macro_features_v4 precomputes every derived global/country column WITH the v4/v5
# timing policy (US-close exogenous shift(1); own/same-session contemporaneous; monthly
# macro ffill+shift(1)). The pooled feature function reads those precomputed columns, so
# the timing policy is identical to v5. We compute the uniform cross-market XM_BLOCK ONCE
# here (determinism + speed: build_panel_pooled reuses it for every security/horizon).
_cols_before = set(df_raw.columns)
t0 = time.time()
df_raw = add_macro_features_v4(df_raw)
print(f'df_raw augmented in {time.time()-t0:.1f}s  →  shape: {df_raw.shape}')

new_cols = sorted(c for c in df_raw.columns if c not in _cols_before)
print(f'Derived macro/global columns added ({len(new_cols)}).')

# Freeze the shared cross-market block (uniform layout reused everywhere).
XM_BLOCK = _xm_block(df_raw)
print(f'XM_BLOCK frozen: {XM_BLOCK.shape[1]} xm_* columns × {XM_BLOCK.shape[0]} dates')

### Leakage audit
Independently re-derive each pooled feature from data available only up to time *t* and compare it against the stored panel value, so any look-ahead leak fails loudly before the heavy runs.

**Scope note.** As in v5, the own-curve and cross-market features are fully re-derived point-in-time; the precomputed macro/global columns are validated as already-lagged stored values rather than re-derived independently — a known scope boundary, not a bug.

In [ ]:
# ── Leakage audit on the pooled feature set (point-in-time recompute) ────────
# Same method as v5: for sampled (date, security) rows, recompute features from
# df_raw.loc[<=t] and compare with the stored panel value. Audits ALL columns,
# including the uniform cross-market xm_* block. Runs on a MULTI-SECURITY pooled
# panel so the audit exercises the categorical/stacking path too. The recompute
# does NOT pass the cached XM_BLOCK (it rebuilds xm from the past slice), which is
# the strict test: stored xm_<col>[t] must equal col[t-1] recomputed from data ≤ t.

def leakage_audit_pooled(panel, df_raw, n_samples=40, rng_seed=config.JULIA_SEED, skip_first=200):
    feat_cols = [c for c in panel.columns if c not in META_COLS]
    eligible = panel.iloc[skip_first:].copy()
    sample = eligible.sample(n=min(n_samples, len(eligible)), random_state=rng_seed)

    TOL = 1e-6
    fail_counts  = {c: 0 for c in feat_cols}
    check_counts = {c: 0 for c in feat_cols}

    for _, row in sample.iterrows():
        t = row['date']
        sec = row['security']
        country, maturity = sec.rsplit('_', 1)
        past = df_raw.loc[df_raw.index <= t]
        if sec not in past.columns or past[sec].isna().all():
            continue
        feats_at_t = _features_for_security_pooled(sec, past, country, maturity).iloc[-1]
        for col in feat_cols:
            if col in CATEGORICAL_COLS:        # string identifiers — not recomputed numerically
                continue
            stored = row[col]
            recomp = feats_at_t.get(col, np.nan)
            check_counts[col] += 1
            if pd.isna(stored) and pd.isna(recomp):
                continue
            if pd.isna(stored) != pd.isna(recomp):
                fail_counts[col] += 1
                continue
            if abs(float(stored) - float(recomp)) > TOL:
                fail_counts[col] += 1

    n_fail = sum(1 for c in feat_cols if fail_counts[c] > 0)
    n_skip = sum(1 for c in feat_cols if check_counts[c] == 0)
    n_pass = len(feat_cols) - n_fail - n_skip
    print(f'=== Pooled leakage audit — {len(feat_cols)} features, {n_samples} sample rows ===\n')
    for col in feat_cols:
        if fail_counts[col] > 0:
            n = check_counts[col]
            print(f'  FAIL  {col}  {n-fail_counts[col]}/{n} OK  ← {fail_counts[col]} mismatch(es)')
    print(f'\n  Summary: {n_pass} PASS / {n_fail} FAIL / {n_skip} SKIP '
          f'(skipped = categoricals + never-sampled)')
    if n_fail == 0:
        print(f'\n[PASS] All {n_pass} checked features leakage-free '
              f'(xm_* = prior close shift(1); own-security contemporaneous; macro ffill+shift(1)).')
    else:
        print(f'\n[FAIL] {n_fail} feature(s) mismatch — fix before the sweep.')
    return n_fail == 0


# Build a small multi-security panel for the audit (NOR + a couple of other countries).
_audit_secs = [s for s in ['NOR_10Y', 'NOR_5Y', 'EUR_10Y', 'SOFR_10Y', 'SWE_5Y'] if s in UNIVERSE]
if len(_audit_secs) < 2:
    _audit_secs = UNIVERSE[:3]
_audit_panel = build_panel_pooled(df_raw, _audit_secs, H, xm_block=XM_BLOCK)
print(f'Audit panel: {_audit_panel.shape}  securities={_audit_secs}  '
      f'({_audit_panel.shape[1]-len(META_COLS)} feature cols)\n')
_audit_ok = leakage_audit_pooled(_audit_panel, df_raw, n_samples=40)
assert _audit_ok, 'Pooled leakage audit failed — fix before proceeding'

## Metrics contract — IDENTICAL shared schema to v5 (the comparability requirement)

Every results row — walk-forward **and** block-CV/LORO, **train** and **oos**, **aggregate-pooled**
and **per-security** — is produced by the single shared function `compute_metrics_row(...)` imported
from **`htb_metrics.py`** (a verbatim extraction of v5's metrics cell). The pooled notebook writes the
**identical** `SHARED_COLS`; it adds pooled-only columns but never renames or drops a shared one. The
result concatenates with `v5_metrics_long.csv` and feeds v5's comparison cell with zero reconciliation.

**Pooled-specific schema conventions (additive — do not affect the shared columns):**

| column | meaning |
|---|---|
| `model_kind` | always `'pooled'` (overridden on every row) |
| `is_pooled` | always `True` |
| `security` | a real security (`NOR_10Y`, …) for per-security breakdown rows; `'__POOL__'` for the aggregate row |
| `country` / `tenor` | the security's, or `'__ALL__'` on the aggregate row |
| `agg_level` *(extra)* | `'aggregate'` or `'per_security'` |
| `use_categoricals` *(extra)* | `True` / `False` — the WI4 with/without-categoricals arm |
| `n_securities` *(extra)* | #securities in the pooled fit producing this row |
| `pca_k` *(extra)* | #cross-market PCA components used in the fit |

**The metric set is exactly v5's** — RW(=0) benchmark (`mse_rw = mean(y²)`), Campbell–Thompson
OOS-R² `= 1 − mse_model/mse_rw`, Clark–West (HAC/Newey–West lag ≈ H−1, one-sided), Diebold–Mariano–
Harvey (HLN small-sample correction, HAC lag ≈ H−1, one-sided), Pesaran–Timmermann (one-sided),
one-sided binomial vs 0.5, and `n_eff ≈ n/H` carried for the overlapping-returns directional caveat.

**MTC families** are kept separate exactly as in v5: a causal walk-forward family and a non-causal
(block-CV ∪ LORO) family, corrected independently. The pooled **headline** is the aggregate OOS over
`{horizon × fold}` (a small test count); per-security rows are **descriptive** (optionally BH-corrected
within `{horizon × tenor × regime}` for the NOR subset to mirror v5, labelled descriptive).

In [ ]:
# ── Metrics binding — the SAME code as v5 (assert the shared schema) ─────────
# compute_metrics_row / clark_west / dm_harvey / pesaran_timmermann / config_hash /
# SHARED_COLS are imported from htb_metrics (verbatim v5 extraction). We assert the
# schema equals the v5 frozen list so the long CSV concatenates with v5's with zero
# reconciliation (acceptance: "assert SHARED_COLS == <v5 list>").
V5_SHARED_COLS = [
    'notebook', 'run_ts', 'model_kind', 'is_pooled', 'validation_scheme', 'target_kind',
    'security', 'country', 'tenor', 'horizon', 'fold', 'regime', 'sample',
    'config_hash', 'feature_count',
    'n_obs', 'dir_acc', 'r2_raw', 'mse_model', 'mse_rw', 'ct_r2_oos',
    'cw_stat', 'cw_pval', 'dmh_stat', 'dmh_pval', 'pt_stat', 'pt_pval',
    'binom_pval', 'n_eff',
]
assert SHARED_COLS == V5_SHARED_COLS, (
    'SHARED_COLS drift between htb_metrics and the v5 contract — comparability broken!\n'
    f'  htb_metrics: {SHARED_COLS}\n  v5 expected : {V5_SHARED_COLS}')


def pooled_metrics_row(y, yhat, H, meta):
    """Shared compute_metrics_row, re-tagged for the pooled model. Identical metric math
    and identical shared columns; only `model_kind`/`is_pooled` are overridden, plus the
    additive pooled-only columns carried on `meta` (agg_level, use_categoricals, …, tau_w)
    and a computed `dir_coverage` (share of scored obs with a nonzero forecast)."""
    row = compute_metrics_row(y, yhat, H, meta)
    row['model_kind'] = 'pooled'
    row['is_pooled'] = True
    for extra in ('agg_level', 'use_categoricals', 'n_securities', 'pca_k', 'tau_w'):
        if extra in meta:
            row[extra] = meta[extra]
    # Directional coverage: share of SCORED obs (finite y & yhat) with a nonzero forecast.
    # ~1.0 for pooled HTBoost, but always emitted for schema parity with the benchmarks.
    _ya = np.asarray(y, dtype=float); _yh = np.asarray(yhat, dtype=float)
    _m = np.isfinite(_ya) & np.isfinite(_yh)
    row['dir_coverage'] = (float(np.mean(np.sign(_yh[_m]) != 0))
                           if int(_m.sum()) >= 5 else np.nan)
    return row


print(f'Metrics bound from htb_metrics (shared with v5).  SHARED_COLS = {len(SHARED_COLS)} cols  [OK]')
print(f'  pooled rows tagged model_kind=pooled, is_pooled=True (shared schema otherwise identical)')

### Cross-market PCA (fit on training rows only)
Compress the large cross-market `xm_*` block to a handful of principal components, fitting the standardizer and PCA on training rows only and applying them to the test rows. Living in the fit path (not the feature builder) keeps the leakage audit valid; the rule (≥`XM_PCA_VAR` of train variance, capped at `XM_PCA_KMAX`) is identical to v5.

In [ ]:
# ── WI1: PCA compression of the cross-market xm_* block — FIT ON TRAIN ONLY ──
# Splits the feature matrix into xm_* (cross-market) vs the rest. Standardises and
# PCA-fits ONLY on the training rows, then transforms train AND test with the SAME
# fitted objects, so test data never touches the PCA fit. The xm_* columns are
# replaced by k components (xmpca_01..xmpca_kk). Pre-committed rule:
#   k = smallest #components explaining ≥ XM_PCA_VAR of TRAIN variance, capped at XM_PCA_KMAX.
# This is in the FIT PATH — deliberately NOT in _features_for_security_v4 (the leakage
# audit recomputes that function from df_raw.loc[<=t] and cannot validate a fold-fit
# transform; putting PCA there would either break the audit or leak).

# Shared cross-market PCA (fit on TRAIN only) — single source of truth.
from src.features.pca import _xm_split, fit_transform_xm_pca


print('PCA fit-path helper defined: fit_transform_xm_pca (fit on TRAIN rows only)')

### HTBoost bridge
Helpers that sanitize the feature frame, build an HTBoost parameter object from the frozen config (threading only fields the installed build exposes, plus the categorical columns), and fit/predict through Julia.

### Note — τ^w aligned to the per-security definition

`extract_weighted_tau` is now imported from the shared `src/models/tau.py` (the per-security v5 helper). The pooled notebook previously **recomputed** τ^w as an importance-weighted geometric mean (its candidate list omitted the package's `gavgtau` field); the canonical helper instead reads the package scalar `gavgtau` directly (paper eq. 19 + fn. 8), using the manual geo-mean only as a logged cross-check. **The reported `tau_w` diagnostic therefore changes** to match the per-security definition. This is a diagnostic-only column — all forecast metric columns in `*_metrics_long.csv` are unchanged.

In [ ]:
# ── juliacall bridge (pooled) — categoricals + frozen config + priortype ─────
# Preserves every v3/v5 invariant:
#   • 4-arg HTBdata(y, x, param, dates);  NO fnames=;  randomizecv=False;  overlap=H-1
#   • categorical string columns (security/country/maturity) cleaned SEPARATELY (kept as
#     strings, NOT filled 0.0) so HTBoost receives them as categoricals
#   • Julia/Optim logger suppressed to Error during HTBfit (v3's PDF was 89% x_tol spew)
#   • RNG re-seeded before every fit (determinism, constraint #6)
# FROZEN_CONFIG threads ONLY loss/modality/nfold/priortype — NO lambda/depth/ntrees
# (modality='accurate' tunes those by internal block CV; pooled prompt §HTBoost-config).

def _sanitize_cols(df):
    rmap = {c: re.sub(r'[^a-zA-Z0-9]', '_', str(c)).strip('_') for c in df.columns}
    return df.rename(columns=rmap), rmap


def _dedup_cols(df):
    seen, new_cols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c] += 1; new_cols.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 0; new_cols.append(c)
    df.columns = new_cols
    return df


def prepare_x_pooled(df, categorical_cols):
    """Clean a feature frame for juliacall. Numeric cols: inf→nan→0.0→float64.
    Categorical (string) cols: left as strings so HTBoost treats them as categoricals."""
    df, _ = _sanitize_cols(df.copy())
    df    = _dedup_cols(df)
    safe_cats = {re.sub(r'[^a-zA-Z0-9]', '_', c).strip('_') for c in categorical_cols}
    num_cols  = [c for c in df.columns if c not in safe_cats]
    df[num_cols] = (df[num_cols].replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0).astype(np.float64))
    for c in (safe_cats & set(df.columns)):
        df[c] = df[c].astype(str)
    return df


def _to_julia_dates(date_series):
    ds = [d.strftime('%Y-%m-%d') for d in pd.to_datetime(date_series)]
    return jl.seval('x -> Date.(x, dateformat"y-m-d")')(ds)


# Introspect the REAL HTBparam field names ONCE (do not assume).
VALID_HTB_FIELDS = set(str(f) for f in jl.seval('collect(string.(fieldnames(typeof(HTBparam()))))'))
for _need in ('nfold', 'modality', 'loss', 'overlap', 'randomizecv'):
    assert _need in VALID_HTB_FIELDS, f'HTBparam has no field {_need!r} — bridge needs it'
_HAS_PRIORTYPE = 'priortype' in VALID_HTB_FIELDS
print(f'HTBparam exposes {len(VALID_HTB_FIELDS)} fields; priortype available: {_HAS_PRIORTYPE}.')


def _build_param(cfg, H):
    """Construct HTBparam from FROZEN_CONFIG, threading ONLY valid + frozen fields.
    No lambda/depth/ntrees (modality tunes them)."""
    kw = dict(loss=cfg.get('loss', LOSS), modality=cfg.get('modality', MODALITY),
              nfold=int(cfg.get('nfold', NFOLD_INTERNAL)), randomizecv=False,
              overlap=int(H - 1), verbose='Off')
    if _HAS_PRIORTYPE and cfg.get('priortype') is not None:
        kw['priortype'] = str(cfg['priortype'])
    bad = [k for k in kw if k not in VALID_HTB_FIELDS and k != 'verbose']
    assert not bad, f'unknown HTBparam keys: {bad}'
    return jl.HTBparam(**kw)


# ── Weighted smoothness τ^w (Task 3) — HTBweightedtau, paper eq.19 ──────────
_LAST_FIT = {}        # most-recent fit's julia objects (for representative retention)


# τ^w now comes from the shared canonical helper (per-security v5 definition:
# reads the package gavgtau scalar, manual geo-mean only as a cross-check).
# This aligns pooled's reported τ^w to per-security; forecast columns are untouched.
from src.models.tau import extract_weighted_tau


def fit_htboost_pooled(y_train, x_train_df, dates_series, x_test_df, cfg, H,
                       categorical_cols, want_importance=False):
    """Fit ONE pooled HTBoost model with the frozen config. Returns
    (output, yhat_train, yhat_test_or_None, importance_dict_or_None)."""
    assert int(H - 1) >= 0
    x_tr     = prepare_x_pooled(x_train_df, categorical_cols)
    x_jl     = jl.DataFrame(x_tr)
    dates_jl = _to_julia_dates(dates_series)
    param    = _build_param(cfg, H)
    data     = jl.HTBdata(y_train, x_jl, param, dates_jl)   # 4-arg, no fnames=

    jl.seval(f'Random.seed!({JULIA_SEED})')                 # determinism per fit
    jl.seval('import Logging; _htb_saved_logger = Logging.global_logger()')
    jl.seval('Logging.global_logger(Logging.SimpleLogger(devnull, Logging.Error))')
    try:
        output = jl.HTBfit(data, param)
    finally:
        jl.seval('Logging.global_logger(_htb_saved_logger)')

    yhat_tr = np.asarray(jl.HTBpredict(x_jl, output), dtype=float)
    yhat_te = None
    if x_test_df is not None:
        x_te    = prepare_x_pooled(x_test_df, categorical_cols)
        yhat_te = np.asarray(jl.HTBpredict(jl.DataFrame(x_te), output), dtype=float)

    importance = None
    tau_w, avgtau = float('nan'), {}
    if want_importance:
        try:
            rel   = jl.HTBrelevance(output, data, verbose=False)
            names = [str(n) for n in rel.fnames]
            fi    = np.asarray(rel.fi, dtype=float)
            importance = dict(zip(names, fi))
        except Exception as e:
            print(f'    [warn] HTBrelevance failed: {repr(e)[:60]}')
        # τ^w is computed from the SAME fitted output (Task 3); only on the importance
        # arm so it adds no cost to other fits. Fail-soft inside the helper.
        tau_w, avgtau = extract_weighted_tau(output, data)
    # Retain THIS fit's julia objects + prepared matrix so the runner can keep ONE
    # representative model for the partial-dependence figure (no extra fits).
    globals()['_LAST_FIT'] = {'output': output, 'data': data, 'x_jl': x_jl,
                              'x_tr': x_tr, 'feature_names': list(x_tr.columns),
                              'imp': importance or {}}
    return output, yhat_tr, yhat_te, importance, (tau_w, avgtau)


print('Pooled bridge defined: fit_htboost_pooled (categoricals + frozen config + seed)')

### Feature taxonomy
Tag every feature column to a bucket (curve, momentum, vol, macro, credit, cross-market, carry/roll, and the cross-sectional identifiers) for the feature-importance report and the macro-vs-curve comparison.

In [ ]:
# ── WI2: pooled feature taxonomy — every column tagged to a bucket ───────────
# Buckets = v5's set MINUS `norway` (disabled) PLUS `categorical` (the cross-sectional
# identifiers). PCA components (xmpca_*) inherit cross_market. Reused for the
# feature-importance-by-bucket report, incl. how much the categoricals are used (WI4).

def bucket_feature(name):
    n = str(name)
    if n in CATEGORICAL_COLS:
        return 'categorical'
    if n == 'carry_roll':
        return 'carry_roll'
    if n.startswith('xm_') or n.startswith('xmpca_'):
        return 'cross_market'
    if n.startswith('ig_') or n.startswith('hy_'):
        return 'credit'
    if n.startswith(('vol_', 'move_', 'vix_', 'v2x_', 'vxn_', 'ovx_', 'gvz_', 'rvx_')):
        return 'vol'
    if n.startswith(('mom_', 'chg_lag')):
        return 'momentum'
    if (n in ('level_zscore', 'swap_ibor_basis') or
            n.startswith(('slope_', 'dev_ma_'))):
        return 'curve'
    # everything else: rates/macro/equity/commodity/breakeven/IBOR exogenous → macro
    return 'macro'


# Build the taxonomy on a representative pooled panel (NOR_10Y + the uniform xm block).
_tax_panel = build_panel_pooled(df_raw, [SMOKE_SEC], H, xm_block=XM_BLOCK)
_tax_feats = [c for c in _tax_panel.columns if c not in (META_COLS - set(CATEGORICAL_COLS))]
# include categoricals as features here (they ARE model inputs in the with-cats arm)
_tax_feats = [c for c in _tax_panel.columns if c not in {'date', 'y', 'level'}]
feature_taxonomy = pd.DataFrame({
    'feature': _tax_feats,
    'bucket':  [bucket_feature(c) for c in _tax_feats],
})
feature_taxonomy.to_csv(f'{OUT_DIR}/pooled_feature_taxonomy.csv', index=False)

print('=== WI2 pooled feature taxonomy ===')
_counts = feature_taxonomy['bucket'].value_counts().sort_index()
print(f'  {"bucket":<14s} {"#features (pre-PCA)":>20s}')
print('  ' + '-' * 36)
for b, c in _counts.items():
    print(f'  {b:<14s} {c:>20d}')
print(f'\n  Total feature columns (pre-PCA, incl. categoricals): {len(_tax_feats)}')
print(f'  cross_market (xm_*) collapse to ≤ {XM_PCA_KMAX} PCA comps in the fit path (v5 rule).')
print(f'  Saved: {OUT_DIR}/pooled_feature_taxonomy.csv')

# Sanity: NO norway bucket should exist (Norway block disabled), categoricals present.
assert 'norway' not in set(feature_taxonomy['bucket']), 'Norway features leaked into pooled set!'
assert (feature_taxonomy['bucket'] == 'categorical').sum() == len(CATEGORICAL_COLS), \
    'categorical identifiers missing from the taxonomy'
print(f'  [OK] no `norway` bucket; {len(CATEGORICAL_COLS)} categorical identifiers present.')

### Walk-forward scorer
Define the pooled runners: one HTBoost fit on the stacked panel per `(horizon × fold)`, scored two ways from the same predictions — an aggregate-pooled row over the whole panel and a per-security breakdown row for every security in the universe. PCA compresses the `xm_*` block on training rows only and the categoricals are re-attached after PCA; the same path serves the causal walk-forward and the non-causal block-CV/LORO schemes.

In [ ]:
# ── WI-P / WI3: ONE pooled fit per (horizon × fold); score aggregate + per-sec ─
# A single HTBoost model is fit on the stacked panel of ALL securities for each
# (horizon × fold), then scored TWO ways from the SAME predictions:
#   (a) aggregate-pooled row over the whole test/train panel  (security='__POOL__')
#   (a2) aggregate over the NORWEGIAN scored set only         (security='__POOL_NOR__')
#   (b) a per-security breakdown row for every NOR tenor (SCORE_UNIVERSE)
# All rows come from the shared pooled_metrics_row (→ SHARED_COLS + additive cols incl.
# tau_w / dir_coverage). PCA compresses the xm_* block, fit on TRAINING rows only (v5
# rule). The same path serves walk-forward (causal) and block-CV/LORO (non-causal).
#
# CHANGES vs the original scorer (all behaviour-preserving for the numbers):
#   • fit_htboost_pooled now returns a 5-tuple (…, tau_pack); tau_pack=(tau_w, avgtau).
#   • tau_w (weighted smoothness, paper eq.19) is threaded onto EVERY metric row of the
#     fit (train + oos, all agg levels) via _score_pooled(tau_w=…).
#   • optional `checkpoint` handle (from the per-horizon helpers) lets each fold be
#     appended to disk + skipped on resume — it changes NO numbers, only durability.
#   • ONE representative fit (H==REP_H, with cats) is retained in REP_FIT for the
#     partial-dependence figure (single model kept, not all of them).

# Representative-fit retention target for the partial-dependence diagnostic (Task 5b).
REP_H   = 21          # representative horizon (NOR_10Y at H=21 in the figure)
REP_FIT = {}          # populated by run_pooled_wf when an H==REP_H cats fit runs


def _cfg_hash(use_categoricals):
    spec = FEAT_SPEC + ('_cats' if use_categoricals else '_nocats')
    return config_hash(FROZEN_CONFIG, extra=spec)


def _prep_fit_matrix(tr, te, numeric, cats):
    """PCA-compress the xm_* numeric block (fit on tr only), re-attach categoricals.
    Returns (x_tr, x_te, pca_info). Indices stay aligned with tr/te."""
    x_tr_num, x_te_num, pca = fit_transform_xm_pca(tr[numeric], te[numeric])
    if cats:
        x_tr = pd.concat([x_tr_num, tr[cats]], axis=1)
        x_te = pd.concat([x_te_num, te[cats]], axis=1)
    else:
        x_tr, x_te = x_tr_num, x_te_num
    return x_tr, x_te, pca


def _score_pooled(panel_part, yhat, H, scheme, fold, regime, sample,
                  use_categoricals, feat_count, pca_k, tau_w=np.nan):
    """Emit the aggregate-pooled row + NOR-pool row + a per-security row for every NOR
    tenor. `tau_w` (weighted smoothness of the fit) is carried on every emitted row."""
    pp = panel_part.reset_index(drop=True).copy()
    pp['_yhat'] = np.asarray(yhat, dtype=float)
    n_secs = int(pp['security'].nunique())
    base = dict(validation_scheme=scheme, target_kind='level_change',
                fold=fold, regime=regime, sample=sample,
                config_hash=_cfg_hash(use_categoricals), feature_count=int(feat_count),
                use_categoricals=bool(use_categoricals), n_securities=n_secs,
                pca_k=int(pca_k), tau_w=float(tau_w))
    rows = []
    # (a) aggregate over the WHOLE pooled panel (all trained securities) → '__POOL__'
    agg_meta = {**base, 'security': '__POOL__', 'country': '__ALL__', 'tenor': '__ALL__',
                'agg_level': 'aggregate'}
    rows.append(pooled_metrics_row(pp['y'].to_numpy(float), pp['_yhat'].to_numpy(float), H, agg_meta))
    # (a2) aggregate over the NORWEGIAN scored securities only → '__POOL_NOR__'. This
    # pooled-over-Norway number is the headline the results section compares against the
    # per-security v5 model. Training is unchanged — we only SLICE predictions to NOR.
    pp_nor = pp[pp['security'].isin(SCORE_UNIVERSE)]
    nor_meta = {**base, 'security': '__POOL_NOR__', 'country': 'NOR', 'tenor': '__ALL__',
                'agg_level': 'aggregate_nor', 'n_securities': int(pp_nor['security'].nunique())}
    rows.append(pooled_metrics_row(pp_nor['y'].to_numpy(float),
                                   pp_nor['_yhat'].to_numpy(float), H, nor_meta))
    # (b) per-security breakdown — SCORED on the Norwegian tenors only (SCORE_UNIVERSE).
    grp = {s: d for s, d in pp.groupby('security')}
    for sec in SCORE_UNIVERSE:
        country, tenor = sec.rsplit('_', 1)
        meta = {**base, 'security': sec, 'country': country, 'tenor': tenor,
                'agg_level': 'per_security'}
        if sec in grp:
            sub = grp[sec]
            rows.append(pooled_metrics_row(sub['y'].to_numpy(float),
                                           sub['_yhat'].to_numpy(float), H, meta))
        else:
            rows.append(pooled_metrics_row(np.array([]), np.array([]), H, meta))
    return rows


def _retain_representative(out, H, fold_name):
    """Keep ONE representative fitted model (H==REP_H, with cats) for the partial-
    dependence figure. Uses _LAST_FIT (set by the bridge). Fail-soft; never raises."""
    try:
        if H == REP_H and not REP_FIT.get('output') and '_LAST_FIT' in globals() and _LAST_FIT:
            REP_FIT.clear()
            REP_FIT.update(_LAST_FIT)
            REP_FIT.update(H=H, fold=fold_name, scheme='walk_forward')
            print(f'  [retain] representative fit kept for partial-dependence: '
                  f'H={H} fold={fold_name}')
    except Exception as _e:
        print(f'  [retain] could not keep representative fit: {repr(_e)[:60]}')


def _checkpoint_fold(checkpoint, scheme, fold_name, use_categoricals, fold_rows, imp_rec):
    """Durably persist one fold (CSV append + imp dump) and mark it done. No-op if no
    checkpoint handle is supplied (e.g. the smoke cell)."""
    if checkpoint is None:
        return
    checkpoint['append'](fold_rows, imp_rec)
    checkpoint['done'].add((str(scheme), str(fold_name), bool(use_categoricals)))


def run_pooled_wf(df_raw, securities, H, cfg, use_categoricals=True,
                  want_importance=False, verbose=True, xm_block=None, checkpoint=None):
    """Walk-forward (causal, expanding window) POOLED fit across all folds at horizon H.
    One fit per fold over the stacked panel; emits aggregate + per-security rows. If a
    `checkpoint` handle is given, each fold is appended to disk immediately and folds
    already on disk are skipped (resume). JULIA_SEED is re-applied per fit, so resumed
    fits are identical to an uninterrupted run."""
    panel = build_panel_pooled(df_raw, securities, H, xm_block=xm_block)
    if len(panel) == 0:
        return [], []
    numeric, cats = feature_columns(panel, use_categoricals)
    rows, imps = [], []
    for fold_name, test_start, test_end, regime in WF_FOLDS:
        key = ('walk_forward', str(fold_name), bool(use_categoricals))
        if checkpoint is not None and key in checkpoint['done']:
            if verbose:
                print(f'  {fold_name}: [resume] already on disk — skip')
            continue
        ts_ts, te_ts = pd.Timestamp(test_start), pd.Timestamp(test_end)
        purge_ts = ts_ts - pd.tseries.offsets.BDay(H - 1)          # OVERLAP = H-1
        tr = panel[panel['date'] < purge_ts].copy()
        te = panel[(panel['date'] >= ts_ts) & (panel['date'] <= te_ts)].copy()
        if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
            if verbose:
                print(f'  {fold_name}: skip (train={len(tr)}, test={len(te)})')
            continue
        try:
            x_tr, x_te, pca = _prep_fit_matrix(tr, te, numeric, cats)
            out, yhat_tr, yhat_te, imp, tau_pack = fit_htboost_pooled(
                tr['y'].to_numpy(float), x_tr, tr['date'], x_te, cfg, H,
                cats, want_importance=want_importance)
        except Exception as e:
            if verbose:
                print(f'  {fold_name}: FAILED — {repr(e)[:90]}')
            continue
        tau_w, avgtau = tau_pack
        fc = x_tr.shape[1]
        fold_rows = (_score_pooled(tr, yhat_tr, H, 'walk_forward', fold_name, regime, 'train',
                                   use_categoricals, fc, pca['k'], tau_w=tau_w)
                     + _score_pooled(te, yhat_te, H, 'walk_forward', fold_name, regime, 'oos',
                                     use_categoricals, fc, pca['k'], tau_w=tau_w))
        rows += fold_rows
        imp_rec = None
        if imp is not None:
            imp_rec = {'horizon': H, 'fold': fold_name, 'use_categoricals': use_categoricals,
                       'imp': imp, 'tau_w': tau_w, 'avgtau': avgtau}
            imps.append(imp_rec)
        _retain_representative(out, H, fold_name)
        _checkpoint_fold(checkpoint, 'walk_forward', fold_name, use_categoricals, fold_rows, imp_rec)
        if verbose:
            _agg = [r for r in fold_rows if r['agg_level'] == 'aggregate'
                    and r['sample'] == 'oos'][-1]
            _nor = [r for r in fold_rows if r['agg_level'] == 'aggregate_nor'
                    and r['sample'] == 'oos'][-1]
            print(f'  {fold_name:<12s} cats={int(use_categoricals)} pca_k={pca["k"]:<2d} '
                  f'OOS_DA(pool)={_agg["dir_acc"]:.3f} OOS_DA(NOR)={_nor["dir_acc"]:.3f} '
                  f'CT-R²(NOR)={_nor["ct_r2_oos"]:+.4f} tau_w={tau_w:.2f} '
                  f'n={_agg["n_obs"]} secs={_agg["n_securities"]}')
    return rows, imps


def _blockcv_entries(scheme):
    """Return [(label, regime, [(start_ts, end_ts), ...]), ...] for the scheme (v5 logic)."""
    if scheme == 'block_cv':
        return [(name, regime, [(pd.Timestamp(s), pd.Timestamp(e))])
                for (name, s, e, regime) in BLOCK_CV_FOLDS]
    if scheme == 'loro':
        out = []
        for regime in dict.fromkeys(r for (_n, _s, _e, r) in BLOCK_CV_FOLDS):
            segs = [(pd.Timestamp(s), pd.Timestamp(e))
                    for (_n, s, e, r) in BLOCK_CV_FOLDS if r == regime]
            out.append((f'LORO_{regime}', regime, segs))
        return out
    raise ValueError(scheme)


def run_pooled_blockcv(df_raw, securities, scheme, H, cfg, use_categoricals=True,
                       want_importance=False, verbose=True, xm_block=None, checkpoint=None):
    """NON-CAUSAL pooled block-CV / LORO: train on data SURROUNDING the held-out block
    (future included), two-sided symmetric purge+embargo gap=(H-1)+EMBARGO around every
    test segment. Reuses FROZEN_CONFIG; emits aggregate + per-security rows. Supports the
    same fold-level checkpoint/resume handle as the walk-forward path."""
    panel = build_panel_pooled(df_raw, securities, H, xm_block=xm_block)
    if len(panel) == 0:
        return [], []
    numeric, cats = feature_columns(panel, use_categoricals)
    gap = (H - 1) + EMBARGO_FOR_H(H)
    dates = panel['date']
    rows, imps = [], []
    for label, regime, segs in _blockcv_entries(scheme):
        key = (str(scheme), str(label), bool(use_categoricals))
        if checkpoint is not None and key in checkpoint['done']:
            if verbose:
                print(f'  {scheme}/{label}: [resume] already on disk — skip')
            continue
        test_mask = pd.Series(False, index=panel.index)
        for (s, e) in segs:
            test_mask |= (dates >= s) & (dates <= e)
        train_mask = ~test_mask
        for (s, e) in segs:                                # symmetric gap, both edges
            lo = s - pd.tseries.offsets.BDay(gap)
            hi = e + pd.tseries.offsets.BDay(gap)
            train_mask &= ~((dates >= lo) & (dates <= hi))
        tr = panel[train_mask].copy()
        te = panel[test_mask].copy()
        if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
            if verbose:
                print(f'  {scheme}/{label}: skip (train={len(tr)}, test={len(te)})')
            continue
        try:
            x_tr, x_te, pca = _prep_fit_matrix(tr, te, numeric, cats)
            out, yhat_tr, yhat_te, imp, tau_pack = fit_htboost_pooled(
                tr['y'].to_numpy(float), x_tr, tr['date'], x_te, cfg, H,
                cats, want_importance=want_importance)
        except Exception as ex:
            if verbose:
                print(f'  {scheme}/{label}: FAILED {repr(ex)[:80]}')
            continue
        tau_w, avgtau = tau_pack
        fc = x_tr.shape[1]
        fold_rows = (_score_pooled(tr, yhat_tr, H, scheme, label, regime, 'train',
                                   use_categoricals, fc, pca['k'], tau_w=tau_w)
                     + _score_pooled(te, yhat_te, H, scheme, label, regime, 'oos',
                                     use_categoricals, fc, pca['k'], tau_w=tau_w))
        rows += fold_rows
        imp_rec = None
        if imp is not None:
            imp_rec = {'horizon': H, 'fold': label, 'use_categoricals': use_categoricals,
                       'imp': imp, 'tau_w': tau_w, 'avgtau': avgtau}
            imps.append(imp_rec)
        _checkpoint_fold(checkpoint, scheme, label, use_categoricals, fold_rows, imp_rec)
        if verbose:
            _agg = [r for r in fold_rows if r['agg_level'] == 'aggregate'
                    and r['sample'] == 'oos'][-1]
            _nor = [r for r in fold_rows if r['agg_level'] == 'aggregate_nor'
                    and r['sample'] == 'oos'][-1]
            print(f'  {scheme:<9s}/{label:<12s} pca_k={pca["k"]:<2d} '
                  f'OOS_DA(pool)={_agg["dir_acc"]:.3f} OOS_DA(NOR)={_nor["dir_acc"]:.3f} '
                  f'tau_w={tau_w:.2f} n={_agg["n_obs"]} secs={_agg["n_securities"]}')
    return rows, imps


print('Pooled runners defined: run_pooled_wf, run_pooled_blockcv '
      '(one fit/fold; aggregate+__POOL_NOR__+per-security rows; tau_w threaded; '
      'PCA fit on train only; optional fold-level checkpoint/resume)')


### Smoke test
A single end-to-end pooled fit on a small multi-country subset to verify the plumbing: the PCA round-trip, the Julia bridge with categoricals, finite non-degenerate predictions, and a real train-vs-OOS gap. Magnitudes are reported, never tuned.

In [ ]:
# ── Smoke test (pooled) — one clean fit on a small multi-country subset ──────
# Fits ONE pooled model (Hiking fold, primary H) over a small multi-country subset with
# PCA on the xm_* block (train rows only) + the security/country/maturity categoricals.
# Verifies plumbing only: PCA round-trip, categoricals reach the matrix, predictions are
# finite/non-degenerate, aggregate + per-security rows are emitted. modality='accurate'
# tunes for OOS via internal block CV, so a large in-sample fit is NOT expected or asserted.
_smoke_secs = [s for s in QUICK_UNIVERSE if s in UNIVERSE]
if len(_smoke_secs) < 3:
    _smoke_secs = UNIVERSE[:6]
_smoke_cfg = {**FROZEN_CONFIG, 'modality': (QUICK_MODALITY if QUICK_MODE else MODALITY)}

_panel = build_panel_pooled(df_raw, _smoke_secs, H, xm_block=XM_BLOCK)
_numeric, _cats = feature_columns(_panel, use_categoricals=True)
_ts, _te_end = pd.Timestamp(WF_FOLDS[-1][1]), pd.Timestamp(WF_FOLDS[-1][2])
_purge = _ts - pd.tseries.offsets.BDay(H - 1)
_tr = _panel[_panel['date'] < _purge].copy()
_te = _panel[(_panel['date'] >= _ts) & (_panel['date'] <= _te_end)].copy()
assert len(_tr) >= MIN_TRAIN_OBS and len(_te) >= MIN_TEST_OBS, \
    f'smoke fold too small (train={len(_tr)}, test={len(_te)})'

_x_tr, _x_te, _pca = _prep_fit_matrix(_tr, _te, _numeric, _cats)
# (a) every PCA component must be non-degenerate
_xmpca_cols = [c for c in _x_tr.columns if c.startswith('xmpca_')]
_xmpca_std = _x_tr[_xmpca_cols].std()
print('PCA component std (train):', ' '.join(f'{s:.3g}' for s in _xmpca_std.values))
assert (_xmpca_std > 1e-9).all(), 'A xmpca_* component is constant — real PCA/bridge bug'
# (b) categoricals reach the model matrix as strings
for c in _cats:
    assert c in _x_tr.columns, f'categorical {c} missing from the model matrix'
    # must be NON-numeric so HTBoost treats it as a categorical (object/str/StringDtype)
    assert not pd.api.types.is_numeric_dtype(_x_tr[c]), \
        f'categorical {c} is numeric — HTBoost would not treat it as categorical'
# (c) curve features survive
assert 'level_zscore' in _x_tr.columns and any(c.startswith('slope_') for c in _x_tr.columns)

print(f'\nSmoke (pooled): securities={_smoke_secs}  (config={_smoke_cfg})')
print(f'  Features pre-PCA: {len(_numeric)} numeric + {len(_cats)} categorical '
      f'(xm_*={sum(1 for f in _numeric if f.startswith("xm_"))})')
print(f'  PCA applied={_pca["applied"]} k={_pca["k"]} (≥{_pca["var_target"]:.0%} train var '
      f'→ cum={_pca["evr_cum_k"]:.3f})  cap binds: '
      f'{(_pca["k"] >= XM_PCA_KMAX) and (_pca["evr_cum_k"] < _pca["var_target"]-1e-9)}')
print(f'  Features post-PCA: {_x_tr.shape[1]}   Train rows={len(_tr)} ({_tr["security"].nunique()} secs)'
      f'  Test rows={len(_te)} ({_te["security"].nunique()} secs)')

t0 = time.time()
_out, _yhat_tr, _yhat_te, _imp, _tau = fit_htboost_pooled(
    _tr['y'].to_numpy(float), _x_tr, _tr['date'], _x_te, _smoke_cfg, H, _cats, want_importance=True)
print(f'  Fit time: {time.time()-t0:.1f}s')

_rows_tr = _score_pooled(_tr, _yhat_tr, H, 'walk_forward', 'Hiking', 'Hiking', 'train',
                         True, _x_tr.shape[1], _pca['k'])
_rows_te = _score_pooled(_te, _yhat_te, H, 'walk_forward', 'Hiking', 'Hiking', 'oos',
                         True, _x_tr.shape[1], _pca['k'])
_agg_tr = [r for r in _rows_tr if r['agg_level'] == 'aggregate'][0]
_agg_te = [r for r in _rows_te if r['agg_level'] == 'aggregate'][0]
_persec_te = [r for r in _rows_te if r['agg_level'] == 'per_security']
_n_eff = _agg_te['n_eff']; _se = 0.5 / np.sqrt(max(_n_eff, 1))
print(f'\n  Train  agg: DirAcc={_agg_tr["dir_acc"]:.3f}  CT-R²={_agg_tr["ct_r2_oos"]:+.4f}  n={_agg_tr["n_obs"]}')
print(f'  OOS    agg: DirAcc={_agg_te["dir_acc"]:.3f}  CT-R²={_agg_te["ct_r2_oos"]:+.4f}  n={_agg_te["n_obs"]}  '
      f'(n_eff≈{_n_eff}, 1 SE≈{_se:.3f}); OOS dist from 0.50 = '
      f'{abs(_agg_te["dir_acc"]-0.5)/_se:.2f} SE (REPORTED, never tuned)')
print(f'  Per-security OOS rows emitted: {len(_persec_te)} (one per universe security)')
_scored = [r for r in _persec_te if r["n_obs"] and r["n_obs"] >= 5]
print(f'    of which scored (n≥5): {len(_scored)}; sample:')
for r in _scored[:6]:
    print(f'      {r["security"]:<10s} OOS DA={r["dir_acc"]:.3f}  CT-R²={r["ct_r2_oos"]:+.4f}  n={r["n_obs"]}')

if _imp:
    _imp_b = {}
    for k, v in _imp.items():
        _imp_b[bucket_feature(k)] = _imp_b.get(bucket_feature(k), 0.0) + v
    print('\n  Smoke feature importance by bucket (train fit; incl. categorical usage):')
    for b, v in sorted(_imp_b.items(), key=lambda kv: -kv[1]):
        print(f'    {b:<14s} {v:6.1f}')

assert np.isfinite(_yhat_te).all(), 'NaN/Inf in OOS preds'
assert np.nanstd(_yhat_te) > 1e-9, 'OOS predictions all identical — fit failed'
assert np.nanstd(_yhat_tr) > 1e-9, 'Train predictions all identical — fit failed'
print('\n[PASS] Smoke pooled — clean fit, PCA round-trip, categoricals wired, '
      'aggregate + per-security rows populated (magnitude reported, not asserted).')

### Resolve the horizon grid
Use the standard horizons and add the long horizons (6m/12m) only when the data supports at least one valid walk-forward fold for the smoke security; otherwise they are logged and skipped. Also resolve the sweep universe and the with/without-categoricals arms (`CAT_ARMS`) the sweeps iterate.

In [ ]:
# ── WI5: resolve the horizon grid + sweep universe (identical gating to v5) ──
# H_GRID always runs. Long horizons (126/252) are included ONLY if the data length
# supports ≥1 walk-forward fold for the smoke security; otherwise logged + skipped.
def _horizon_supported(h):
    panel = build_panel_pooled(df_raw, [SMOKE_SEC], h, xm_block=XM_BLOCK)
    if len(panel) == 0:
        return False
    for _, ts, te, _r in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(ts), pd.Timestamp(te)
        purge = ts_ts - pd.tseries.offsets.BDay(h - 1)
        ntr = (panel['date'] < purge).sum()
        nte = ((panel['date'] >= ts_ts) & (panel['date'] <= te_ts)).sum()
        if ntr >= MIN_TRAIN_OBS and nte >= MIN_TEST_OBS:
            return True
    return False


if QUICK_MODE:
    HORIZONS = list(QUICK_HGRID)
    SWEEP_UNIVERSE = [s for s in QUICK_UNIVERSE if s in UNIVERSE]
    print(f'[QUICK_MODE] HORIZONS={HORIZONS}  SWEEP_UNIVERSE={len(SWEEP_UNIVERSE)} secs')
else:
    HORIZONS = list(H_GRID)
    for h in H_GRID_LONG:
        if _horizon_supported(h):
            HORIZONS.append(h)
            print(f'  long horizon H={h}: data supports it → included')
        else:
            print(f'  long horizon H={h}: insufficient data per fold → SKIPPED (logged)')
    SWEEP_UNIVERSE = list(UNIVERSE)

HORIZONS = sorted(set(HORIZONS))
# WI4 arms: always fit WITH categoricals; optionally also WITHOUT (the value-of-cats delta).
CAT_ARMS = [True] + ([False] if RUN_NO_CATS else [])

print(f'\nFinal horizon grid: {HORIZONS}')
print(f'Sweep universe ({len(SWEEP_UNIVERSE)} securities), '
      f'{len({s.rsplit("_",1)[0] for s in SWEEP_UNIVERSE})} countries')
print(f'Categorical arms (WI4): {CAT_ARMS}  (True=with cats, False=without)')
print(f'Frozen config (no forced lambda): {FROZEN_CONFIG}')
assert FROZEN_CONFIG is not None

### Per-horizon sweep — helpers
Decision 3 splits the boosting work by horizon. `run_wf_horizon(h)` and `run_blockcv_horizon(h)` each run the full fold set for ONE horizon, **overwrite only that horizon's file** (`pooled_metrics_wf_H{h}.csv` / `pooled_metrics_blockcv_H{h}.csv`), and persist importances. Run the per-horizon cells below individually — each is independently checkpointed and re-runnable. Training uses the full universe; scoring is restricted to the Norwegian tenors. `JULIA_SEED` stays pinned so the split reproduces the single-sweep result.

In [ ]:
# ── Per-horizon sweep helpers — INDEPENDENT, re-runnable, FOLD-LEVEL CHECKPOINTED ─
# Each helper runs the full fold set for ONE horizon and OWNS one per-horizon CSV.
# CRASH-SAFETY (Task 6): every fold's rows are APPENDED to the per-horizon CSV the moment
# it is scored (flushed + fsync'd), and the importance/tau records are re-dumped per fold
# — so a crash mid-horizon never loses a completed fit. RESUME: on entry we read which
# (scheme, fold, cat-arm) fits already exist on disk and SKIP refitting them, so re-running
# a horizon cell after a crash continues where it stopped and never duplicates rows.
# JULIA_SEED stays pinned and is re-applied before EVERY fit (in the bridge), so a resumed
# fit is byte-identical to an uninterrupted one — resume changes no number. The downstream
# assemble cell de-duplicates on stable keys as a final safety net.
import pickle, glob, os

from src.config import MACHINE_ID
from src.utils.gitpush import push_results
WF_H_CSV    = lambda h: f'{OUT_DIR}/pooled_metrics_wf_H{h}__{MACHINE_ID}.csv'
WF_IMP_PKL  = lambda h: f'{OUT_DIR}/pooled_wf_imps_H{h}__{MACHINE_ID}.pkl'
BLK_H_CSV   = lambda h: f'{OUT_DIR}/pooled_metrics_blockcv_H{h}__{MACHINE_ID}.csv'
BLK_IMP_PKL = lambda h: f'{OUT_DIR}/pooled_blockcv_imps_H{h}__{MACHINE_ID}.pkl'

# Fixed column order for incremental appends (the header is written ONCE per file, so
# every appended chunk must present columns in exactly this order). = shared schema +
# the additive pooled-only columns that every pooled row carries.
CKPT_COLS = list(SHARED_COLS) + ['agg_level', 'use_categoricals', 'n_securities',
                                 'pca_k', 'dir_coverage', 'tau_w']


def _done_keys_from_csv(path):
    """Set of (validation_scheme, fold, use_categoricals) fits already scored in `path`."""
    if not os.path.exists(path):
        return set()
    try:
        d = pd.read_csv(path, usecols=['validation_scheme', 'fold', 'use_categoricals'])
    except Exception as e:
        print(f'  [resume] could not read {path} ({repr(e)[:50]}) — treating as empty')
        return set()
    return {(str(a), str(b), bool(c)) for a, b, c in
            zip(d['validation_scheme'], d['fold'], d['use_categoricals'])}


def _load_pkl_list(path):
    if not os.path.exists(path):
        return []
    try:
        with open(path, 'rb') as f:
            return list(pickle.load(f))
    except Exception as e:
        print(f'  [resume] could not load {path} ({repr(e)[:50]}) — starting imps fresh')
        return []


def _make_checkpoint(csv_path, pkl_path):
    """Build a checkpoint handle: a 'done' set seeded from disk + an 'append' callback
    that durably writes one fold's rows (CSV append, flush+fsync) and re-dumps the
    accumulated imps (flush+fsync). Returned dict is passed straight to the runners."""
    done = _done_keys_from_csv(csv_path)
    imps_accum = _load_pkl_list(pkl_path)

    def _append(fold_rows, imp_rec):
        if fold_rows:
            d = pd.DataFrame(fold_rows).reindex(columns=CKPT_COLS)
            write_header = not os.path.exists(csv_path)
            with open(csv_path, 'a', newline='') as fh:
                d.to_csv(fh, header=write_header, index=False)
                fh.flush(); os.fsync(fh.fileno())
        if imp_rec is not None:
            imps_accum.append(imp_rec)
            with open(pkl_path, 'wb') as fh:
                pickle.dump(imps_accum, fh)
                fh.flush(); os.fsync(fh.fileno())

    return {'done': done, 'append': _append, 'imps': imps_accum}


def run_wf_horizon(h):
    """Walk-forward pooled sweep for ONE horizon over CAT_ARMS, fold-level checkpointed.
    Appends each fold to WF_H_CSV(h) immediately and resumes by skipping fits already on
    disk. Safe to re-run after a crash (continues; never duplicates rows)."""
    if not RUN_WALK_FORWARD:
        print('[skip] RUN_WALK_FORWARD = False'); return pd.DataFrame()
    csv_path, pkl_path = WF_H_CSV(h), WF_IMP_PKL(h)
    ckpt = _make_checkpoint(csv_path, pkl_path)
    pre_done = set(ckpt['done'])
    if pre_done:
        print(f'[WF H={h}] resume: {len(pre_done)} fit(s) already on disk → will skip them')
    _t0 = time.time()
    for use_cats in CAT_ARMS:
        print(f'[WF H={h}] cats={use_cats}')
        try:
            run_pooled_wf(df_raw, SWEEP_UNIVERSE, h, FROZEN_CONFIG,
                          use_categoricals=use_cats, want_importance=use_cats,
                          verbose=True, xm_block=XM_BLOCK, checkpoint=ckpt)
        except Exception as e:
            print(f'    FAILED {repr(e)[:90]}'); continue
    new_keys = ckpt['done'] - pre_done
    out = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
    print(f'[WF H={h}] done: {len(new_keys)} new fit(s), {len(pre_done)} skipped; '
          f'{len(out)} rows total in {(time.time()-_t0)/60:.1f} min  (→ {csv_path})')
    if config.AUTO_PUSH:
        push_results([csv_path], f'results: pooled_v5 walk_forward H{h} @ {MACHINE_ID}')
    return out


def run_blockcv_horizon(h):
    """Block-CV / LORO pooled sweep for ONE horizon (non-causal, cats only), fold-level
    checkpointed + resumable exactly like the walk-forward helper."""
    _schemes = (['block_cv'] if RUN_BLOCK_CV else []) + (['loro'] if RUN_LORO else [])
    if not _schemes:
        print('[skip] RUN_BLOCK_CV = RUN_LORO = False'); return pd.DataFrame()
    csv_path, pkl_path = BLK_H_CSV(h), BLK_IMP_PKL(h)
    ckpt = _make_checkpoint(csv_path, pkl_path)
    pre_done = set(ckpt['done'])
    if pre_done:
        print(f'[block-CV H={h}] resume: {len(pre_done)} fit(s) already on disk → will skip them')
    _t0 = time.time()
    for scheme in _schemes:
        print(f'[block-CV H={h}] scheme={scheme} cats=True')
        try:
            run_pooled_blockcv(df_raw, SWEEP_UNIVERSE, scheme, h, FROZEN_CONFIG,
                               use_categoricals=True, want_importance=True,
                               verbose=True, xm_block=XM_BLOCK, checkpoint=ckpt)
        except Exception as e:
            print(f'    FAILED {repr(e)[:90]}'); continue
    new_keys = ckpt['done'] - pre_done
    out = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
    print(f'[block-CV H={h}] done: {len(new_keys)} new fit(s), {len(pre_done)} skipped; '
          f'{len(out)} rows total in {(time.time()-_t0)/60:.1f} min  (→ {csv_path})')
    if config.AUTO_PUSH:
        push_results([csv_path], f'results: pooled_v5 blockcv H{h} @ {MACHINE_ID}')
    return out


print('Per-horizon sweep helpers (fold-level checkpoint + resume) defined: '
      'run_wf_horizon(h), run_blockcv_horizon(h)')
print('  - each fold is appended + fsync\'d immediately; re-running a horizon resumes '
      'from disk and never duplicates rows (assemble de-dups as a final guard).')


In [ ]:
# ── Finest assignable unit (Task 6a): ONE (horizon, scheme, cat-arm) per call ─────────
# run_arm reuses the SAME per-horizon machine-stamped CSVs and the SAME _make_checkpoint /
# runners as run_wf_horizon / run_blockcv_horizon — it just dispatches a SINGLE arm instead
# of looping all cat-arms (WF) or all schemes (block-CV). Multiple arms for one horizon
# therefore append to the same WF_H_CSV(h) / BLK_H_CSV(h), and resume still skips already-done
# (validation_scheme, fold, use_categoricals) fits. No checkpoint machinery and no fit logic
# is rebuilt here; only the granularity at which work is dispatched across machines changes.

def run_arm(h, scheme, use_cats):
    """Run exactly ONE arm = (horizon h, scheme, use_categoricals). Independently
    re-runnable/resumable; appends to the existing per-horizon machine-stamped CSV and skips
    fits already on disk. Returns the full per-horizon CSV as a DataFrame."""
    if scheme == 'walk_forward':
        if not RUN_WALK_FORWARD:
            print('[skip] RUN_WALK_FORWARD = False'); return pd.DataFrame()
        csv_path, pkl_path = WF_H_CSV(h), WF_IMP_PKL(h)
    elif scheme in ('block_cv', 'loro'):
        if (scheme == 'block_cv' and not RUN_BLOCK_CV) or (scheme == 'loro' and not RUN_LORO):
            print(f'[skip] {scheme} disabled'); return pd.DataFrame()
        csv_path, pkl_path = BLK_H_CSV(h), BLK_IMP_PKL(h)
    else:
        raise ValueError(f'unknown scheme {scheme!r} (expected walk_forward / block_cv / loro)')

    ckpt = _make_checkpoint(csv_path, pkl_path)
    pre_done = set(ckpt['done'])
    if pre_done:
        print(f'[arm H={h} {scheme} cats={use_cats}] resume: {len(pre_done)} fit(s) on disk -> skip')
    _t0 = time.time()
    try:
        if scheme == 'walk_forward':
            run_pooled_wf(df_raw, SWEEP_UNIVERSE, h, FROZEN_CONFIG,
                          use_categoricals=use_cats, want_importance=use_cats,
                          verbose=True, xm_block=XM_BLOCK, checkpoint=ckpt)
        else:
            run_pooled_blockcv(df_raw, SWEEP_UNIVERSE, scheme, h, FROZEN_CONFIG,
                               use_categoricals=use_cats, want_importance=use_cats,
                               verbose=True, xm_block=XM_BLOCK, checkpoint=ckpt)
    except Exception as e:
        print(f'    FAILED {repr(e)[:90]}')
    new_keys = ckpt['done'] - pre_done
    out = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
    print(f'[arm H={h} {scheme} cats={use_cats}] {len(new_keys)} new, {len(pre_done)} skipped; '
          f'{len(out)} rows total in {(time.time()-_t0)/60:.1f} min  (-> {csv_path})')
    if config.AUTO_PUSH:
        push_results([csv_path], f'results: pooled_v5 {scheme} H{h} cats={use_cats} @ {MACHINE_ID}')
    return out


def arm_grid(horizons=None):
    """The canonical full arm set the per-horizon cells cover, now individually addressable:
    walk_forward x {cats-off, cats-on}, block_cv x {cats-on}, loro x {cats-on}."""
    hs = HORIZONS if horizons is None else horizons
    arms = []
    for h in hs:
        for uc in CAT_ARMS:
            arms.append((h, 'walk_forward', uc))
        if RUN_BLOCK_CV:
            arms.append((h, 'block_cv', True))
        if RUN_LORO:
            arms.append((h, 'loro', True))
    return arms


print('Single-arm driver defined: run_arm(h, scheme, use_cats) + arm_grid(horizons=None)')
print('  one arm per cell across machines, e.g.:  run_arm(21, "walk_forward", True)')


In [ ]:
# ── THIS MACHINE'S ARM ASSIGNMENT (Task 6a) ───────────────────────────────────────────
# Give this machine its slice — one arm per tuple. Each arm is independently
# re-runnable/resumable and appends to its per-horizon machine-stamped CSV; re-running skips
# (scheme, fold, cats) fits already on disk. Empty by default so a top-to-bottom run does NOT
# launch a multi-hour sweep — assign deliberately, e.g. ARMS = [(21, 'walk_forward', True)].
# (See arm_grid() for the full canonical set if one machine should take everything.)
ARMS = []        # e.g. [(21, 'walk_forward', True)]  -> "H21, walk_forward, cats-on"

if not ARMS:
    print('[arms] none assigned — set ARMS to this machine\'s slice (see arm_grid()).')
for _h, _scheme, _uc in ARMS:
    if _h in HORIZONS:
        run_arm(_h, _scheme, _uc)
    else:
        print(f'[H={_h}] not in HORIZONS={HORIZONS} — skipped')


### Walk-forward pooled sweep — split by horizon
One boosting cell per horizon (`H_GRID` = 1, 5, 21, 63 always; `H_GRID_LONG` = 126, 252 gated on data length via membership in `HORIZONS`). Each cell appends its rows to its own per-horizon file and is safe to re-run in isolation.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# REPRO_HANDSHAKE — paste IDENTICALLY at the top of BOTH GB notebooks, AFTER the
# data load + fit-function defs + (pooled) XM_BLOCK, but BEFORE the heavy sweep.
# Part A prints a deterministic FINGERPRINT to eyeball-diff across machines.
# Part B runs a ~1-min pilot fit through the REAL fit→score→persist path.
# ═══════════════════════════════════════════════════════════════════════════════
import os as _os, json as _json, hashlib as _hashlib, pickle as _pickle
import numpy as _np, pandas as _pd
import src.config as _C                       # read config WITHOUT shadowing notebook globals
from src.evaluation.metrics import SHARED_COLS as _SHARED

def _sha16(b):                                 # stable, unsalted SHA-256 (first 16 hex)
    return _hashlib.sha256(b).hexdigest()[:16]

def _hash_df(df):
    return _sha16(_pd.util.hash_pandas_object(df, index=True).values.tobytes())

def _resolve(*names):                          # first present, non-None global
    g = globals()
    for n in names:
        if n in g and g[n] is not None:
            return n, g[n]
    return None, None

# ── Part A — deterministic fingerprint ─────────────────────────────────────────
_L = ['FINGERPRINT:']
if 'df_raw' in globals():
    _dfr = globals()['df_raw']
    # DATA-INTEGRITY GATE: raw (pre-augmentation) values-only hash, portable across CPU arch
    # (round_trip float parse + deterministic column order). df_raw.hash16 below is INFORMATIONAL:
    # the ~88 derived columns differ at ULP (~1e-15) across x64/arm64 SIMD, which does not move results.
    try:
        from src.data.bloomberg import load_data as _load_data
        _raw_vfp = _sha16(_pd.util.hash_pandas_object(_load_data(), index=False).values.tobytes())
    except Exception as _e:
        _raw_vfp = f'(unavailable: {repr(_e)[:40]})'
    _L += [f'  df_raw.shape         = {tuple(_dfr.shape)}',
           f'  df_raw_values.hash16 = {_raw_vfp}   [DATA GATE - must match]',
           f'  df_raw.hash16        = {_hash_df(_dfr)}   [informational - ULP arch-variant in derived cols]',
           f'  df_raw.index_min     = {_dfr.index.min()}',
           f'  df_raw.index_max     = {_dfr.index.max()}']
else:
    _L.append('  df_raw              = (unavailable — load data before the handshake)')

_pn_name, _pn = _resolve('panel', 'PANEL', 'pooled_panel', 'df_panel')
if isinstance(_pn, _pd.DataFrame):
    _L += [f'  panel({_pn_name}).shape  = {tuple(_pn.shape)}',
           f'  panel.hash16        = {_hash_df(_pn)}']
else:
    _L.append('  panel               = (not built at handshake point — informational)')

_cfg = {'JULIA_SEED': _C.JULIA_SEED, 'H_GRID': _C.H_GRID, 'WF_FOLDS': _C.WF_FOLDS,
        'BLOCK_CV_FOLDS': _C.BLOCK_CV_FOLDS, 'NOR_TENORS': _C.NOR_TENORS,
        'MIN_TRAIN_OBS': _C.MIN_TRAIN_OBS, 'MIN_TEST_OBS': _C.MIN_TEST_OBS,
        'ALPHA_MT': _C.ALPHA_MT, 'XM_PCA_ENABLE': _C.XM_PCA_ENABLE,
        'XM_PCA_VAR': _C.XM_PCA_VAR, 'XM_PCA_KMAX': _C.XM_PCA_KMAX,
        'FROZEN_CONFIG': globals().get('FROZEN_CONFIG')}
_L.append(f'  config.hash16       = {_sha16(_json.dumps(_cfg, sort_keys=True, default=str).encode())}')

_meta = globals().get('META_COLS', {'date', 'security', 'y', 'level'})
_feat = None
if isinstance(_pn, _pd.DataFrame):
    _feat = sorted(c for c in _pn.columns if c not in _meta)
else:
    for _nm in ('NOR_FEATURE_COLS', 'FEATURE_COLS', 'feature_cols', 'feat_cols'):
        if globals().get(_nm):
            _feat = sorted(globals()[_nm]); break
if _feat is not None:
    _L += [f'  feature_spec.n      = {len(_feat)}',
           f'  feature_spec.hash16 = {_sha16(_json.dumps(_feat, sort_keys=True).encode())}']
else:
    _L.append('  feature_spec        = (unavailable at handshake point)')

import numpy as _vnp, pandas as _vpd, scipy as _vsp
_L += [f'  MACHINE_ID          = {_C.MACHINE_ID}',
       f'  versions            = numpy {_vnp.__version__} / pandas {_vpd.__version__} / scipy {_vsp.__version__}']
_jl = globals().get('jl')
try:
    if _jl is not None:
        _jv = str(_jl.seval('string(VERSION)'))
        try:
            _hv = str(_jl.seval('string(pkgversion(HybridTreeBoosting))'))
        except Exception:
            _hv = '?'
        _L.append(f'  julia / HTBoost     = julia {_jv} / HybridTreeBoosting {_hv}')
    else:
        _L.append('  julia / HTBoost     = (jl not loaded)')
except Exception as _e:
    _L.append(f'  julia / HTBoost     = (unreachable: {repr(_e)[:50]})')
print('\n'.join(_L))
print('  ⇒ eyeball-diff df_raw / config / feature_spec (and panel) hashes vs your partner')
print('    BEFORE the full sweep; any mismatch ⇒ NOT the same inputs ⇒ stop and reconcile.\n')

# ── Part B — pilot fit (~8 min) through the REAL fit→score→persist path ─────────
# EXACTLY ONE fit (1 security × 1 horizon × 1 fold) at the REAL FROZEN_CONFIG — NO
# modality / nfold / nofullsample override — so it is byte-path-identical to a sweep
# fit, just run once. Cheap by SCOPE, not by modality: τ^w (gavgtau) is only the
# reported quantity when HTBoost actually tunes (modality ∈ {compromise, accurate});
# a fast/fastest fit would emit a DIFFERENT, untuned τ^w and give false cross-machine
# confidence. This one-time ~8-min per-machine cost is the point — it exercises the
# tuning code path where any cross-machine FP divergence would surface. Pilot outputs
# go to results/<OUT_DIR>/_pilot/... (gitignored).
import time as _time
_TAG       = globals().get('NOTEBOOK_TAG', 'nb')
_PILOT_DIR = f'{OUT_DIR}/_pilot/{_TAG}__{_C.MACHINE_ID}'
_os.makedirs(_PILOT_DIR, exist_ok=True)
_SEC, _H   = 'NOR_10Y', 21

def _need(field, ok):
    assert ok, f'PILOT FAIL: missing/empty output field -> {field}'

_cfg_pilot = globals().get('FROZEN_CONFIG')    # the REAL sweep config; do NOT override
_need('FROZEN_CONFIG set (run the regularization cell first)',
      isinstance(_cfg_pilot, dict) and len(_cfg_pilot) > 0)
_modality  = _cfg_pilot.get('modality') or globals().get('MODALITY')
_FOLD      = WF_FOLDS[-1]                       # last (Hiking, expanding) fold → most training data
_t0        = _time.time()

_rows, _imp_present, _tau_w, _sidecars = [], False, None, []
if 'run_pooled_wf' in globals():               # ── POOLED notebook ──
    _pcsv = f'{_PILOT_DIR}/pilot_pooled_wf_H{_H}.csv'
    _ppkl = f'{_PILOT_DIR}/pilot_pooled_wf_imps_H{_H}.pkl'
    for _f in (_pcsv, _ppkl):
        if _os.path.exists(_f):
            _os.remove(_f)
    _ck = _make_checkpoint(_pcsv, _ppkl)
    _saved_wf = list(WF_FOLDS)
    try:
        globals()['WF_FOLDS'] = [_FOLD]
        _rows, _imps = run_pooled_wf(df_raw, [_SEC], _H, _cfg_pilot,
                                     use_categoricals=False, want_importance=True,
                                     verbose=False, xm_block=globals().get('XM_BLOCK'),
                                     checkpoint=_ck)
    finally:
        globals()['WF_FOLDS'] = _saved_wf
    _oos = [r for r in _rows if r.get('sample') == 'oos']
    _imp_present = len(_imps) > 0
    _tau_w = (_oos[0].get('tau_w') if _oos else None)
    _sidecars = [_pcsv]
elif 'run_security_v5' in globals():           # ── PER-SECURITY notebook ──
    _saved_wf = list(WF_FOLDS)
    try:
        globals()['WF_FOLDS'] = [_FOLD]
        _rows, _imps, _taus = run_security_v5(_SEC, df_raw, _H, _cfg_pilot,
                                              want_importance=True, want_tau=True, verbose=False)
    finally:
        globals()['WF_FOLDS'] = _saved_wf
    _oos = [r for r in _rows if r.get('sample') == 'oos']
    _pcsv = f'{_PILOT_DIR}/pilot_metrics_H{_H}.csv'
    _pd.DataFrame(_rows).to_csv(_pcsv, index=False)
    if _imps:
        with open(f'{_PILOT_DIR}/pilot_imps_H{_H}.pkl', 'wb') as _fh:
            _pickle.dump(_imps, _fh)
    if _taus:
        _pd.DataFrame([{k: v for k, v in t.items() if k != 'table'} for t in _taus]
                      ).to_csv(f'{_PILOT_DIR}/pilot_tau_H{_H}.csv', index=False)
    _imp_present = len(_imps) > 0
    _tau_w = (_taus[0].get('tau_w') if _taus else None)
    _sidecars = [_pcsv]
else:
    raise RuntimeError('PILOT FAIL: neither run_pooled_wf nor run_security_v5 in scope — '
                       'paste the handshake AFTER the fit-function cells.')

assert len(WF_FOLDS) == 5, 'PILOT FAIL: WF_FOLDS not restored to 5 folds after the pilot slice'
_mins = (_time.time() - _t0) / 60.0

# ── assertions: every expected field present + non-empty; fail LOUDLY by name ──
_need('pilot produced rows', len(_rows) > 0)
_need('oos row', len(_oos) > 0)
_row = _oos[0]
for _c in _SHARED:                             # schema completeness: every shared field present
    _need(f'SHARED_COLS[{_c}]', (_c in _row) and (_row[_c] is not None))
for _c in ('n_obs', 'dir_acc', 'r2_raw', 'mse_model', 'mse_rw', 'ct_r2_oos', 'binom_pval'):
    _v = _row.get(_c)                          # core metrics must be real (not NaN)
    _need(f'{_c} (NaN)', _v is not None and not (isinstance(_v, float) and _np.isnan(_v)))
_need('n_obs > 0', int(_row['n_obs']) > 0)
_need('tau_w', _tau_w is not None and not (isinstance(_tau_w, float) and _np.isnan(_tau_w)))
_need('importance record', _imp_present)
for _sc in _sidecars:
    _need(f'sidecar CSV {_sc}', _os.path.exists(_sc) and _os.path.getsize(_sc) > 0)

print(f'PILOT OK (modality={_modality}, ~{_mins:.0f}m): all output fields present  '
      f'({_SEC} H={_H} fold={_FOLD[0]}; rows={len(_rows)}; tau_w={_tau_w:.3f}; '
      f'sidecars={[_os.path.basename(s) for s in _sidecars]}; under {_PILOT_DIR}/)')


In [ ]:
# Walk-forward pooled sweep — horizon H=1 (independent · re-runnable · overwrites its own file)
_H_CELL = 1
if RUN_WALK_FORWARD and _H_CELL in HORIZONS:
    _df_h = run_wf_horizon(_H_CELL)
    print(f'walk-forward H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or RUN_WALK_FORWARD=False — skipped')

In [ ]:
# Walk-forward pooled sweep — horizon H=5 (independent · re-runnable · overwrites its own file)
_H_CELL = 5
if RUN_WALK_FORWARD and _H_CELL in HORIZONS:
    _df_h = run_wf_horizon(_H_CELL)
    print(f'walk-forward H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or RUN_WALK_FORWARD=False — skipped')

In [ ]:
# Walk-forward pooled sweep — horizon H=21 (independent · re-runnable · overwrites its own file)
_H_CELL = 21
if RUN_WALK_FORWARD and _H_CELL in HORIZONS:
    _df_h = run_wf_horizon(_H_CELL)
    print(f'walk-forward H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or RUN_WALK_FORWARD=False — skipped')

In [ ]:
# Walk-forward pooled sweep — horizon H=63 (independent · re-runnable · overwrites its own file)
_H_CELL = 63
if RUN_WALK_FORWARD and _H_CELL in HORIZONS:
    _df_h = run_wf_horizon(_H_CELL)
    print(f'walk-forward H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or RUN_WALK_FORWARD=False — skipped')

In [ ]:
# Walk-forward pooled sweep — horizon H=126 (independent · re-runnable · overwrites its own file)
_H_CELL = 126
if RUN_WALK_FORWARD and _H_CELL in HORIZONS:
    _df_h = run_wf_horizon(_H_CELL)
    print(f'walk-forward H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or RUN_WALK_FORWARD=False — skipped')

In [ ]:
# Walk-forward pooled sweep — horizon H=252 (independent · re-runnable · overwrites its own file)
_H_CELL = 252
if RUN_WALK_FORWARD and _H_CELL in HORIZONS:
    _df_h = run_wf_horizon(_H_CELL)
    print(f'walk-forward H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or RUN_WALK_FORWARD=False — skipped')

### Block-CV / LORO pooled sweep — split by horizon
Non-causal learnability diagnostic, one boosting cell per horizon, same split/checkpoint discipline. Never enters Part C or the causal headline.

In [ ]:
# Block-CV / LORO pooled sweep — horizon H=1 (non-causal · independent · re-runnable)
_H_CELL = 1
if (RUN_BLOCK_CV or RUN_LORO) and _H_CELL in HORIZONS:
    _df_h = run_blockcv_horizon(_H_CELL)
    print(f'block-CV/LORO H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or block-CV/LORO disabled — skipped')

In [ ]:
# Block-CV / LORO pooled sweep — horizon H=5 (non-causal · independent · re-runnable)
_H_CELL = 5
if (RUN_BLOCK_CV or RUN_LORO) and _H_CELL in HORIZONS:
    _df_h = run_blockcv_horizon(_H_CELL)
    print(f'block-CV/LORO H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or block-CV/LORO disabled — skipped')

In [ ]:
# Block-CV / LORO pooled sweep — horizon H=21 (non-causal · independent · re-runnable)
_H_CELL = 21
if (RUN_BLOCK_CV or RUN_LORO) and _H_CELL in HORIZONS:
    _df_h = run_blockcv_horizon(_H_CELL)
    print(f'block-CV/LORO H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or block-CV/LORO disabled — skipped')

In [ ]:
# Block-CV / LORO pooled sweep — horizon H=63 (non-causal · independent · re-runnable)
_H_CELL = 63
if (RUN_BLOCK_CV or RUN_LORO) and _H_CELL in HORIZONS:
    _df_h = run_blockcv_horizon(_H_CELL)
    print(f'block-CV/LORO H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or block-CV/LORO disabled — skipped')

In [ ]:
# Block-CV / LORO pooled sweep — horizon H=126 (non-causal · independent · re-runnable)
_H_CELL = 126
if (RUN_BLOCK_CV or RUN_LORO) and _H_CELL in HORIZONS:
    _df_h = run_blockcv_horizon(_H_CELL)
    print(f'block-CV/LORO H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or block-CV/LORO disabled — skipped')

In [ ]:
# Block-CV / LORO pooled sweep — horizon H=252 (non-causal · independent · re-runnable)
_H_CELL = 252
if (RUN_BLOCK_CV or RUN_LORO) and _H_CELL in HORIZONS:
    _df_h = run_blockcv_horizon(_H_CELL)
    print(f'block-CV/LORO H={_H_CELL}: df shape {_df_h.shape}')
else:
    print(f'[H={_H_CELL}] not in HORIZONS={HORIZONS} or block-CV/LORO disabled — skipped')

### Part C — Hiking-period OOS export
Export out-of-sample predictions for the hiking regime from one pooled fit, alongside a par carry/roll decomposition, for downstream trading analysis. Export only: carry/roll is not a model target and these predictions never feed model selection; causal walk-forward only — block-CV/LORO predictions never enter this cell. Writes the long per-`(date, security)` file plus signal/realized/carry matrices pivoted on `security` (the panel is multi-country, so a tenor-only pivot would collide across countries).

In [ ]:
# ── PART C (pooled) — Hiking-first OOS trading export, parametrized by H ──────
# EXPORT-ONLY. Carry/roll is NOT a model target (it may be a point-in-time feature for
# parity with v5). Predictions are OUT-OF-SAMPLE from ONE pooled fit (the model never
# trains on the Hiking window). Causal walk-forward only — block-CV/LORO predictions
# NEVER enter this cell. The v3-style long/short Sharpe is deliberately NOT produced as a
# headline (it contradicted the forecasting result); any book belongs in an appendix
# behind honest diagnostics.
HIKE_START, HIKE_END = pd.Timestamp('2022-01-01'), pd.Timestamp('2026-12-31')
PARTC_HORIZONS = HORIZONS if not QUICK_MODE else list(QUICK_HGRID)


def _spot_clamped(country, T, df):
    cols = {MATURITY_YEARS[m]: df[f'{country}_{m}']
            for m in MATURITY_YEARS if f'{country}_{m}' in df.columns}
    if not cols:
        return pd.Series(np.nan, index=df.index)
    pts = sorted(cols)
    if T <= pts[0]:
        return cols[pts[0]]
    if T >= pts[-1]:
        return cols[pts[-1]]
    for i in range(len(pts) - 1):
        if pts[i] <= T <= pts[i + 1]:
            w = (T - pts[i]) / (pts[i + 1] - pts[i])
            return cols[pts[i]] * (1 - w) + cols[pts[i + 1]] * w
    return pd.Series(np.nan, index=df.index)


def carry_roll_decomp_h(sec, df, country, maturity, h):
    """Par carry+roll on the SPOT curve, scaled to horizon h. EXPORT-ONLY."""
    M = MATURITY_YEARS.get(maturity, np.nan)
    Hy = h / 252.0
    if np.isnan(M):
        return None
    s_M   = df[sec]
    s_MmH = _spot_clamped(country, M - Hy, df)
    roll  = s_M - s_MmH
    ibor_col = COUNTRY_PRIMARY_IBOR.get(country, (None, None))[0]
    short = (df[ibor_col].copy() if ibor_col and ibor_col in df.columns
             else pd.Series(np.nan, index=df.index))
    carry = (s_M - short) * Hy
    return pd.DataFrame({'date': df.index, 's_spot_M': s_M.values,
                         's_spot_M_minus_H': s_MmH.values, 'short_funding': short.values,
                         'carry': carry.values, 'roll_down': roll.values,
                         'carry_roll_total': (carry + roll).values})


def pooled_hiking_export(df_raw, securities, h, cfg):
    """ONE pooled fit; train < Hiking purge; predict the whole Hiking OOS panel.
    Returns a per-(date,security) frame with predictions + par carry/roll decomposition."""
    panel = build_panel_pooled(df_raw, securities, h, xm_block=XM_BLOCK)
    if len(panel) == 0:
        return None
    numeric, cats = feature_columns(panel, use_categoricals=True)
    purge_ts = HIKE_START - pd.tseries.offsets.BDay(h - 1)
    tr = panel[panel['date'] < purge_ts].copy()
    te = panel[(panel['date'] >= HIKE_START) & (panel['date'] <= HIKE_END)].copy()
    if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
        return None
    x_tr, x_te, _ = _prep_fit_matrix(tr, te, numeric, cats)
    _, _, yhat, _, _ = fit_htboost_pooled(tr['y'].to_numpy(float), x_tr, tr['date'],
                                       x_te, cfg, h, cats)
    out = te[['date', 'security', 'country', 'maturity', 'level', 'carry_roll', 'y']].copy()
    out = out.rename(columns={'maturity': 'tenor', 'level': 'level_t', 'y': 'y_true'})
    out['regime']  = 'Hiking'
    out['y_pred']  = np.asarray(yhat, dtype=float)
    out['signal']  = np.sign(out['y_pred'])
    out['hit']     = (np.sign(out['y_pred']) == np.sign(out['y_true'])).astype(int)
    out['horizon'] = h
    out['n_train'] = len(tr)
    # Decision 2/4: trained on the FULL panel above; EXPORT/score the NOR tenors only.
    # Slice AFTER predictions are attached so yhat stays aligned with the full panel.
    out = out[out['country'] == 'NOR'].copy()
    if len(out) == 0:
        return None
    # attach par carry/roll decomposition per security
    decs = []
    for sec in out['security'].unique():
        c, m = sec.rsplit('_', 1)
        d = carry_roll_decomp_h(sec, df_raw, c, m, h)
        if d is not None:
            d['security'] = sec
            decs.append(d)
    if decs:
        dec_all = pd.concat(decs, ignore_index=True)
        out = out.merge(dec_all, on=['date', 'security'], how='left')
    return out


if not RUN_WALK_FORWARD:
    print('[Part C skipped] RUN_WALK_FORWARD = False')
else:
    for h in PARTC_HORIZONS:
        _purge = HIKE_START - pd.tseries.offsets.BDay(h - 1)
        print(f'PART C (pooled) — Hiking OOS export, H={h}  (train < {_purge.date()})')
        r = pooled_hiking_export(df_raw, SWEEP_UNIVERSE, h, FROZEN_CONFIG)
        if r is None or len(r) == 0:
            print('  (no Hiking fold at this H)'); continue
        _cols = ['date', 'security', 'country', 'tenor', 'regime', 'level_t', 'carry_roll',
                 'y_true', 'y_pred', 'signal', 'hit', 'horizon', 'n_train', 's_spot_M',
                 's_spot_M_minus_H', 'short_funding', 'carry', 'roll_down', 'carry_roll_total']
        _cols = [c for c in _cols if c in r.columns]
        r[_cols].sort_values(['date', 'security']).to_csv(
            f'{OUT_DIR}/pooled_hiking_predictions_H{h}.csv', index=False)
        # matrix exports for the trading sim — pivot on SECURITY, not tenor: the
        # pooled panel is multi-country, so a tenor-only pivot would COLLIDE across
        # countries (NOR_10Y vs USD_10Y). Columns sorted deterministically.
        def _pivot_save(val, fname):
            p = r.pivot_table(index='date', columns='security', values=val, aggfunc='first')
            p = p.reindex(columns=sorted(p.columns))
            p.to_csv(f'{OUT_DIR}/{fname}')
        _pivot_save('y_pred',           f'pooled_hiking_signal_matrix_H{h}.csv')
        _pivot_save('y_true',           f'pooled_hiking_realized_matrix_H{h}.csv')
        _pivot_save('carry_roll_total', f'pooled_hiking_carry_matrix_H{h}.csv')
        # per-security and NOR-subset hit summaries
        _bysec = r.groupby('security')['hit'].mean().sort_values(ascending=False)
        print(f'  OOS rows={len(r)}  securities={r["security"].nunique()}  '
              f'mean DirAcc={r["hit"].mean():.3f}')
        _nor = r[r['country'] == 'NOR']
        if len(_nor):
            print(f'  NOR subset DirAcc={_nor["hit"].mean():.3f} (n={len(_nor)}) — aligns with v5 Part C')
        print(f'  saved pooled_hiking_predictions_H{h}.csv + signal/realized/carry matrices (pivot on security)')
    print('\n[PART C COMPLETE] OOS Hiking predictions + par carry/roll exported per H '
          '(pooled, causal walk-forward only). DV01 excluded (duration guard). No long/short '
          'Sharpe presented as model skill.')

### Assemble per-horizon sweep results
Concatenate every per-horizon file into the full combined `df_wf` / `df_block` (+ importances) before the multiple-testing correction. Only generation is split by horizon; the correction and aggregation below see the full table, exactly as before.

In [ ]:
# ── Assemble per-horizon sweep files → df_wf / df_block (+ importances) ───────
# Decision 3: only the GENERATION is split by horizon; the MTC/aggregation downstream
# sees the FULL combined table. Concatenate every per-horizon CSV that exists (re-running
# one horizon refreshes only its file). Importances are reloaded for the bucket report.
import pickle, glob  # standalone-safe (also imported in the helpers cell)


def _concat_horizon_csvs(pattern):
    files = sorted(glob.glob(pattern))
    parts = []
    for f in files:
        try:
            d = pd.read_csv(f)
            if len(d):
                parts.append(d)
        except Exception as e:
            print(f'  [warn] could not read {f}: {repr(e)[:60]}')
    return (pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()), files


def _load_imps(pattern):
    out = []
    for f in sorted(glob.glob(pattern)):
        try:
            with open(f, 'rb') as _fh:
                out += pickle.load(_fh)
        except Exception as e:
            print(f'  [warn] could not load {f}: {repr(e)[:60]}')
    return out


df_wf,    _wf_files  = _concat_horizon_csvs(f'{OUT_DIR}/pooled_metrics_wf_H*__*.csv')
df_block, _blk_files = _concat_horizon_csvs(f'{OUT_DIR}/pooled_metrics_blockcv_H*__*.csv')
wf_imps    = _load_imps(f'{OUT_DIR}/pooled_wf_imps_H*__*.pkl')
block_imps = _load_imps(f'{OUT_DIR}/pooled_blockcv_imps_H*__*.pkl')

print('=== assembled per-horizon sweep files ===')
print(f'  walk-forward : {len(_wf_files)} horizon files → df_wf {df_wf.shape}')
print(f'  block-CV/LORO: {len(_blk_files)} horizon files → df_block {df_block.shape}')
print(f'  importances  : wf={len(wf_imps)}  block={len(block_imps)}')
if len(df_wf):
    print(f'  horizons present (WF): {sorted(df_wf["horizon"].unique().tolist())}')
    _nor = df_wf[(df_wf['agg_level'] == 'aggregate_nor') & (df_wf['sample'] == 'oos')]
    print(f'  NOR-pool OOS rows (aggregate_nor): {len(_nor)}')

### Assemble results and apply multiple-testing correction
Concatenate all rows into the tidy long CSV (mergeable with v5's `v5_metrics_long.csv`) and apply Bonferroni and BH-FDR within two separate families — causal walk-forward (forecastability) and non-causal block-CV/LORO (learnability) — since they test different hypotheses.

In [ ]:
# ── WI3/WI4/WI6/WI-MTC: assemble the tidy long CSV + apply MTC (separate families) ─
# The shared schema (SHARED_COLS) is preserved so this CSV concatenates with
# v5_metrics_long.csv with zero reconciliation. df_wf / df_block are the FULL combined
# tables assembled from the per-horizon files — MTC sees everything at once (Decision 3:
# only the generation is split by horizon, the correction is not). Two aggregate levels
# are carried: '__POOL__' (whole panel, agg_level='aggregate') and '__POOL_NOR__' (the
# Norwegian scored set, agg_level='aggregate_nor' — the v5 comparison headline). The
# per-security rows (NOR tenors only) are DESCRIPTIVE.
_parts = [d for d in (df_wf, df_block) if len(d) > 0]
df_all = pd.concat(_parts, ignore_index=True) if _parts else pd.DataFrame(columns=SHARED_COLS)

# Ensure every shared column exists (comparability contract).
for c in SHARED_COLS:
    if c not in df_all.columns:
        df_all[c] = np.nan
# Additive pooled-only + MTC columns.
for c, default in [('agg_level', ''), ('use_categoricals', True), ('n_securities', np.nan),
                   ('pca_k', np.nan), ('dir_coverage', np.nan), ('tau_w', np.nan),
                   ('reject_bonferroni', False), ('reject_fdr_bh', False),
                   ('mtc_N', np.nan), ('mtc_family', ''), ('mtc_scope', '')]:
    if c not in df_all.columns:
        df_all[c] = default


# De-duplicate on the stable identity keys BEFORE any MTC — a horizon produced across
# several crash-resumed passes (Task 6) must NOT double-count cells in the family sizes.
_DEDUP_KEYS = ['validation_scheme', 'horizon', 'fold', 'regime', 'sample',
               'agg_level', 'security', 'use_categoricals', 'config_hash']
_have_keys = [k for k in _DEDUP_KEYS if k in df_all.columns]
if len(df_all) and _have_keys:
    _n0 = len(df_all)
    df_all = df_all.drop_duplicates(subset=_have_keys, keep='last').reset_index(drop=True)
    if len(df_all) != _n0:
        print(f'  [dedup] removed {_n0 - len(df_all)} duplicate row(s) on stable keys '
              f'(resume safety) -> {len(df_all)} rows')


def apply_mtc_headline(df, schemes, label, agg='aggregate'):
    """Bonferroni + BH-FDR over OOS binomial p of the WITH-categoricals aggregate rows at
    the given agg_level, in ONE family. Writes reject_*/mtc_N/mtc_family/mtc_scope in
    place; returns (N, n_bonf, n_bh)."""
    mask = ((df['sample'] == 'oos') & (df['agg_level'] == agg)
            & (df['use_categoricals'] == True) & df['validation_scheme'].isin(schemes)
            & df['binom_pval'].notna())
    idx = df.index[mask]
    if len(idx) == 0:
        return 0, 0, 0
    pvals = df.loc[idx, 'binom_pval'].to_numpy(float)
    rej_b, _, _, _ = multipletests(pvals, alpha=ALPHA_MT, method='bonferroni')
    rej_h, _, _, _ = multipletests(pvals, alpha=ALPHA_MT, method='fdr_bh')
    df.loc[idx, 'reject_bonferroni'] = rej_b
    df.loc[idx, 'reject_fdr_bh']     = rej_h
    df.loc[idx, 'mtc_N']             = len(idx)
    df.loc[idx, 'mtc_family']        = label
    df.loc[idx, 'mtc_scope']         = 'headline_' + agg
    return len(idx), int(rej_b.sum()), int(rej_h.sum())


# HEADLINE (the v5 comparison) = NOR-pool aggregate. Whole-pool aggregate is kept as a
# secondary family so every aggregate row still carries reject flags.
N_wf,  b_wf,  h_wf  = apply_mtc_headline(df_all, ['walk_forward'], 'wf:NORpool{H×fold}', 'aggregate_nor')
N_wfp, b_wfp, h_wfp = apply_mtc_headline(df_all, ['walk_forward'], 'wf:pool{H×fold}', 'aggregate')
N_bl,  b_bl,  h_bl  = apply_mtc_headline(df_all, ['block_cv', 'loro'], 'noncausal:NORpool{scheme×H×fold}', 'aggregate_nor')
N_blp, b_blp, h_blp = apply_mtc_headline(df_all, ['block_cv', 'loro'], 'noncausal:pool{scheme×H×fold}', 'aggregate')

# Descriptive NOR per-security BH (mirrors v5's per-tenor view; NOT the headline).
_nor_mask = ((df_all['sample'] == 'oos') & (df_all['agg_level'] == 'per_security')
             & (df_all['validation_scheme'] == 'walk_forward')
             & (df_all['use_categoricals'] == True) & (df_all['country'] == 'NOR')
             & df_all['binom_pval'].notna())
_nor_idx = df_all.index[_nor_mask]
N_nor = b_nor = h_nor = 0
if len(_nor_idx):
    _p = df_all.loc[_nor_idx, 'binom_pval'].to_numpy(float)
    _rb, _, _, _ = multipletests(_p, alpha=ALPHA_MT, method='bonferroni')
    _rh, _, _, _ = multipletests(_p, alpha=ALPHA_MT, method='fdr_bh')
    df_all.loc[_nor_idx, 'reject_bonferroni'] = _rb
    df_all.loc[_nor_idx, 'reject_fdr_bh']     = _rh
    df_all.loc[_nor_idx, 'mtc_N']             = len(_nor_idx)
    df_all.loc[_nor_idx, 'mtc_family']        = 'descriptive:NOR{H×tenor×regime}'
    df_all.loc[_nor_idx, 'mtc_scope']         = 'descriptive_per_security'
    N_nor, b_nor, h_nor = len(_nor_idx), int(_rb.sum()), int(_rh.sum())

# Order columns: shared first, then extras.
_extra = [c for c in df_all.columns if c not in SHARED_COLS]
df_all = df_all[[c for c in SHARED_COLS if c in df_all.columns] + _extra]

# Write to v5's POOLED_CSV_PATH (so v5's comparison cell merges it) AND a local copy.
# canonical; run final assemble on one machine to avoid clobber
os.makedirs(V5_OUT_DIR, exist_ok=True)
df_all.to_csv(POOLED_CSV_PATH, index=False)
df_all.to_csv(LOCAL_LONG_CSV, index=False)

print('=== pooled_metrics_long.csv written ===')
print(f'  rows={len(df_all)}  cols={df_all.shape[1]}  (shared={len(SHARED_COLS)})')
print(f'  → {POOLED_CSV_PATH}  (v5 comparison cell reads here)')
print(f'  → {LOCAL_LONG_CSV}  (local copy)')
print(f'  HEADLINE wf NOR-pool    MTC: N={N_wf}  Bonferroni={b_wf}  BH-FDR={h_wf}  (aggregate_nor {{H×fold}})')
print(f'  secondary wf whole-pool MTC: N={N_wfp} Bonferroni={b_wfp} BH-FDR={h_wfp}')
print(f'  HEADLINE noncausal NORpool : N={N_bl}  Bonferroni={b_bl}  BH-FDR={h_bl}')
print(f'  secondary noncausal pool   : N={N_blp} Bonferroni={b_blp} BH-FDR={h_blp}')
print(f'  DESCRIPTIVE NOR per-sec    : N={N_nor} Bonferroni={b_nor} BH-FDR={h_nor} (not the headline)')

# Concatenation smoke-check against v5 (if present): shared columns line up exactly.
_v5_csv = f'{V5_OUT_DIR}/v5_metrics_long.csv'
if os.path.exists(_v5_csv):
    _v5 = pd.read_csv(_v5_csv)
    _miss = [c for c in SHARED_COLS if c not in _v5.columns]
    _shared_ok = not _miss
    _cat = pd.concat([_v5[[c for c in SHARED_COLS if c in _v5.columns]],
                      df_all[[c for c in SHARED_COLS]]], ignore_index=True)
    print(f'\n  [concat check] v5_metrics_long.csv present: shared cols align = {_shared_ok}; '
          f'combined OOS rows = {len(_cat)} (pooled {len(df_all)} + v5 {len(_v5)})')
else:
    print(f'\n  [concat check] {_v5_csv} not present yet — run v5 to merge; schema is v5-identical.')

### Value of the cross-sectional categoricals
Pooled-only diagnostic: pair the with- and without-categoricals OOS results and report the delta, answering whether the `security`/`country`/`maturity` identifiers add forecasting skill. Descriptive only — it never re-selects the model.

In [ ]:
# ── WI4: value of the cross-sectional categoricals (with − without) ──────────
# Do the security/country/maturity identifiers add anything OOS? Compare the AGGREGATE
# walk-forward OOS of the with-cats vs without-cats arms, per horizon and overall. This
# is a REPORTED delta — never a selection (both arms use the same FROZEN config).
oos_agg = df_all[(df_all['validation_scheme'] == 'walk_forward')
                 & (df_all['sample'] == 'oos') & (df_all['agg_level'] == 'aggregate_nor')]

cat_delta = pd.DataFrame()
if len(oos_agg) and oos_agg['use_categoricals'].nunique() == 2:
    piv = oos_agg.pivot_table(index=['horizon', 'fold', 'regime'],
                              columns='use_categoricals',
                              values=['dir_acc', 'ct_r2_oos'], aggfunc='first')
    piv.columns = [f'{a}_{"cats" if b else "nocats"}' for a, b in piv.columns]
    piv = piv.reset_index()
    if 'dir_acc_cats' in piv and 'dir_acc_nocats' in piv:
        piv['d_dir_acc'] = piv['dir_acc_cats'] - piv['dir_acc_nocats']
    if 'ct_r2_oos_cats' in piv and 'ct_r2_oos_nocats' in piv:
        piv['d_ct_r2'] = piv['ct_r2_oos_cats'] - piv['ct_r2_oos_nocats']
    cat_delta = piv
    cat_delta.to_csv(f'{OUT_DIR}/pooled_value_of_categoricals.csv', index=False)
    print('=== WI4 value-of-categoricals — aggregate walk-forward OOS (with − without) ===')
    _show = [c for c in ['horizon', 'fold', 'regime', 'dir_acc_cats', 'dir_acc_nocats',
                         'd_dir_acc', 'ct_r2_oos_cats', 'ct_r2_oos_nocats', 'd_ct_r2']
             if c in cat_delta.columns]
    print(cat_delta[_show].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    if 'd_dir_acc' in cat_delta:
        print(f'\n  mean Δ DirAcc (cats − no-cats): {cat_delta["d_dir_acc"].mean():+.4f}')
    if 'd_ct_r2' in cat_delta:
        print(f'  mean Δ Campbell–Thompson R²:    {cat_delta["d_ct_r2"].mean():+.4f}')
    print('  Reading: Δ≈0 ⇒ the cross-sectional identifiers add no OOS skill (consistent with the')
    print('  null); Δ>0 across MULTIPLE regimes ⇒ genuine borrowed cross-sectional strength.')
    print(f'  Saved: {OUT_DIR}/pooled_value_of_categoricals.csv')
else:
    print('[WI4] value-of-categoricals needs both arms (RUN_NO_CATS=True) and OOS rows — skipped.')

### Report — train-vs-OOS gap, horizon/regime aggregates, feature buckets
Pair the pooled train and OOS metrics with an explicit overfit-gap column, aggregate by horizon and regime, place block-CV OOS next to walk-forward OOS under the asymmetric interpretation rule, and bucket feature importance for the report.

In [ ]:
# ── WI3/WI6 report — aggregate train-vs-OOS, horizon/regime aggregates, buckets ─
# Headline uses the AGGREGATE, WITH-categoricals rows (the pooled model's whole-panel
# performance). Per-security aggregates are summarised separately. Walk-forward is the
# causal finding; block-CV is the non-causal diagnostic.
_AGG = (df_all['agg_level'] == 'aggregate_nor') & (df_all['use_categoricals'] == True)
wf_agg = df_all[_AGG & (df_all['validation_scheme'] == 'walk_forward')]


def _pair_train_oos(d):
    keys = ['horizon', 'fold', 'regime']
    piv = d.pivot_table(index=keys, columns='sample',
                        values=['dir_acc', 'r2_raw', 'ct_r2_oos', 'n_obs'], aggfunc='first')
    piv.columns = [f'{a}_{b}' for a, b in piv.columns]
    piv = piv.reset_index()
    if {'dir_acc_train', 'dir_acc_oos'} <= set(piv.columns):
        piv['gap_dir_acc'] = piv['dir_acc_train'] - piv['dir_acc_oos']
    return piv


master_wf = _pair_train_oos(wf_agg) if len(wf_agg) else pd.DataFrame()
if len(master_wf):
    master_wf.to_csv(f'{OUT_DIR}/pooled_master_overfit_gap.csv', index=False)
    print('=== Pooled NOR aggregate train-vs-OOS (walk-forward, with cats) ===')
    _show = [c for c in ['horizon', 'fold', 'regime', 'dir_acc_train', 'dir_acc_oos',
                         'gap_dir_acc', 'ct_r2_oos_oos', 'n_obs_oos'] if c in master_wf.columns]
    print(master_wf[_show].to_string(index=False, float_format=lambda x: f'{x:.3f}'))

oos_wf = wf_agg[wf_agg['sample'] == 'oos']
agg_h = agg_r = pd.DataFrame()
if len(oos_wf):
    print('\n=== Aggregate by HORIZON (walk-forward OOS, pooled NOR) ===')
    agg_h = oos_wf.groupby('horizon').agg(
        n=('dir_acc', 'size'), mean_DA=('dir_acc', 'mean'),
        mean_CT_R2=('ct_r2_oos', 'mean'),
        bonf=('reject_bonferroni', 'sum'), bh=('reject_fdr_bh', 'sum'))
    print(agg_h.to_string(float_format=lambda x: f'{x:.3f}'))
    agg_h.to_csv(f'{OUT_DIR}/pooled_agg_by_horizon.csv')
    print('\n=== Aggregate by REGIME (walk-forward OOS, pooled NOR) ===')
    agg_r = oos_wf.groupby('regime').agg(
        n=('dir_acc', 'size'), mean_DA=('dir_acc', 'mean'), mean_CT_R2=('ct_r2_oos', 'mean'))
    print(agg_r.to_string(float_format=lambda x: f'{x:.3f}'))
    agg_r.to_csv(f'{OUT_DIR}/pooled_agg_by_regime.csv')

# Per-security OOS distribution (descriptive) — how the pooled model does name-by-name.
persec_oos = df_all[(df_all['agg_level'] == 'per_security')
                    & (df_all['validation_scheme'] == 'walk_forward')
                    & (df_all['use_categoricals'] == True) & (df_all['sample'] == 'oos')
                    & df_all['dir_acc'].notna()]
if len(persec_oos):
    print('\n=== Per-security OOS DirAcc (descriptive) — top/bottom by mean across folds ===')
    _bysec = persec_oos.groupby('security')['dir_acc'].mean().sort_values(ascending=False)
    print('  top:   ' + ', '.join(f'{s}={v:.3f}' for s, v in _bysec.head(5).items()))
    print('  bottom:' + ', '.join(f'{s}={v:.3f}' for s, v in _bysec.tail(5).items()))
    print(f'  mean per-security OOS DirAcc: {persec_oos["dir_acc"].mean():.4f}  '
          f'(NOR subset: {persec_oos[persec_oos["country"]=="NOR"]["dir_acc"].mean():.4f})')

# Feature importance by bucket (incl. categorical usage) — walk-forward, with cats.
def _agg_importance(imps):
    recs = []
    for r in imps:
        if not r.get('use_categoricals', True):
            continue
        b = {}
        for feat, val in r['imp'].items():
            bk = bucket_feature(feat)
            b[bk] = b.get(bk, 0.0) + float(val)
        b['horizon'] = r['horizon']
        recs.append(b)
    if not recs:
        return pd.DataFrame()
    return pd.DataFrame(recs).fillna(0.0).groupby('horizon').mean(numeric_only=True)


imp_buckets = _agg_importance(wf_imps)
if len(imp_buckets):
    print('\n=== Feature importance by bucket (walk-forward, with cats, per horizon) ===')
    print(imp_buckets.to_string(float_format=lambda x: f'{x:.1f}'))
    imp_buckets.to_csv(f'{OUT_DIR}/pooled_importance_buckets.csv')
    if 'categorical' in imp_buckets.columns:
        print(f'  mean categorical-identifier importance: {imp_buckets["categorical"].mean():.1f} '
              f'(how much the pooled model leans on security/country/maturity)')

# Block-CV vs walk-forward aggregate OOS (asymmetric interpretation).
def _scheme_oos_agg(scheme):
    d = df_all[(df_all['sample'] == 'oos') & (df_all['agg_level'] == 'aggregate_nor')
               & (df_all['use_categoricals'] == True) & (df_all['validation_scheme'] == scheme)]
    return d.groupby('regime')['dir_acc'].mean() if len(d) else pd.Series(dtype=float)


cmp_block = None
wf_m = _scheme_oos_agg('walk_forward'); bl_m = _scheme_oos_agg('block_cv'); lo_m = _scheme_oos_agg('loro')
if len(wf_m):
    cmp_block = pd.DataFrame({'walk_forward_OOS': wf_m})
    if len(bl_m):
        cmp_block['block_cv_OOS'] = bl_m
        cmp_block['gap_block_minus_wf'] = cmp_block['block_cv_OOS'] - cmp_block['walk_forward_OOS']
    if len(lo_m):
        cmp_block['loro_OOS'] = lo_m
    cmp_block = cmp_block.reset_index()
    cmp_block.to_csv(f'{OUT_DIR}/pooled_blockcv_vs_walkforward.csv', index=False)
    print('\n=== Block-CV / LORO OOS vs walk-forward OOS (aggregate mean DirAcc by regime) ===')
    print(cmp_block.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# ── self-contained pooled_v5_report.md ───────────────────────────────────────
def _md_table(df, floatfmt='{:.3f}', maxrows=40):
    if df is None or len(df) == 0:
        return '_(no rows)_\n'
    d = df.copy()
    if isinstance(d.index, pd.MultiIndex) or d.index.name:
        d = d.reset_index()
    d = d.head(maxrows)
    cols = list(d.columns)
    def fmt(v):
        return (floatfmt.format(v) if isinstance(v, (int, float, np.floating))
                and not isinstance(v, bool) and pd.notna(v) else str(v))
    out = ['| ' + ' | '.join(map(str, cols)) + ' |',
           '| ' + ' | '.join('---' for _ in cols) + ' |']
    for _, r in d.iterrows():
        out.append('| ' + ' | '.join(fmt(r[c]) for c in cols) + ' |')
    return '\n'.join(out) + '\n'


_L = []
_L.append('# HTBoost pooled v5 — Results report\n')
_L.append(f'_Generated {RUN_TS} · notebook={NOTEBOOK_TAG} · config_hash={FROZEN_HASH}_\n')
_L.append(f'ONE pooled model per (horizon × fold) over {len(SWEEP_UNIVERSE)} securities '
          f'({len({s.rsplit("_",1)[0] for s in SWEEP_UNIVERSE})} countries). '
          f'Frozen config `{FROZEN_CONFIG}` (no forced lambda; modality tunes the rest). '
          f'PCA xm_* → ≤{XM_PCA_KMAX} comps (≥{XM_PCA_VAR:.0%} train var) — identical rule to v5.\n')
_L.append('\n## Decision rule (ex ante)\n')
_L.append('Null confirmed if aggregate OOS DirAcc is noise-consistent and no {horizon×fold} cell '
          'survives MTC. A genuine lead requires MTC survival AND multi-regime positivity '
          '(not Hiking-only). Block-CV is a non-causal learnability diagnostic, never a forecast.\n')
_L.append('\n## Walk-forward — aggregate train vs OOS (with categoricals)\n')
_L.append(_md_table(master_wf[_show] if len(master_wf) else None))
_L.append('\n## Aggregate by horizon (walk-forward OOS)\n')
_L.append(_md_table(agg_h))
_L.append('\n## Aggregate by regime (walk-forward OOS)\n')
_L.append(_md_table(agg_r))
_L.append('\n## Value of categoricals (with − without), aggregate OOS — WI4\n')
_L.append(_md_table(cat_delta if 'cat_delta' in dir() and len(cat_delta) else None))
_L.append('\n## Multiple-testing survivors (separate families)\n')
_L.append(f'- HEADLINE walk-forward `{{horizon×fold}}`: N={N_wf}, Bonferroni={b_wf}, BH-FDR={h_wf}\n')
_L.append(f'- HEADLINE non-causal `{{scheme×horizon×fold}}`: N={N_bl}, Bonferroni={b_bl}, BH-FDR={h_bl}\n')
_L.append(f'- DESCRIPTIVE NOR per-security `{{horizon×tenor×regime}}`: N={N_nor}, '
          f'Bonferroni={b_nor}, BH-FDR={h_nor} (not the headline)\n')
_L.append('\n## Feature importance by bucket (per horizon)\n')
_L.append(_md_table(imp_buckets, floatfmt='{:.1f}'))
_L.append('\n## Block-CV / LORO vs walk-forward (asymmetric interpretation)\n')
_L.append('block-CV is a NON-causal learnability diagnostic, never a forecast and never in Part C. '
          'block-CV≈0.50 ⇒ stronger null; block-CV>wf while wf≈0.50 ⇒ future-info/size artefact '
          '(non-stationarity), not skill.\n')
_L.append(_md_table(cmp_block))
with open(f'{OUT_DIR}/pooled_v5_report.md', 'w') as _f:
    _f.write('\n'.join(_L))
print(f'\n[REPORT] wrote {OUT_DIR}/pooled_v5_report.md (+ supporting CSVs)')

### Diagnostic figures (tau^w, partial dependence, importance buckets, NOR OOS heatmaps)

In [ ]:
# ── Diagnostic figures (run AFTER assembly; safe to re-run; does NOT re-fit) ──
# Generates and SAVES four labelled diagnostics from the emitted results (+ the retained
# representative model for the partial-dependence panel). EVERY figure is guarded: a
# missing input prints a warning and skips that panel rather than erroring. matplotlib
# only — no new heavy dependencies.
import os, glob, pickle
import numpy as np, pandas as pd
import matplotlib
matplotlib.use(matplotlib.get_backend())  # keep current backend; figures are also saved to disk
import matplotlib.pyplot as plt

FIG_DIR = f'{OUT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)
_saved_figs = []


def _finish(fig, fname, caption):
    """Save a figure to FIG_DIR, display inline, and print its caption."""
    path = f'{FIG_DIR}/{fname}'
    fig.tight_layout()
    fig.savefig(path, dpi=140, bbox_inches='tight')
    _saved_figs.append(path)
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    print(f'[fig] {path}\n      {caption}\n')


# Make the results frame available whether or not cell 56 is in memory.
try:
    _df = df_all.copy()
except NameError:
    try:
        _df = pd.read_csv(LOCAL_LONG_CSV)
        print(f'[figures] df_all not in memory — loaded {LOCAL_LONG_CSV}')
    except Exception as _e:
        _df = pd.DataFrame()
        print(f'[figures][warn] no df_all and could not load {LOCAL_LONG_CSV}: {repr(_e)[:60]}')

# Importance records (for the taxonomy-bucket figure): memory first, else reload pkls.
try:
    _imps = list(wf_imps)
except NameError:
    _imps = []
    for _f in sorted(glob.glob(f'{OUT_DIR}/pooled_wf_imps_H*__*.pkl')):
        try:
            with open(_f, 'rb') as _fh:
                _imps += pickle.load(_fh)
        except Exception:
            pass


# ── (a) tau^w summary — NOR per-security smoothness by tenor and horizon ─────
try:
    if not len(_df) or 'tau_w' not in _df.columns:
        raise ValueError('no tau_w column in results')
    nor = _df[(_df['agg_level'] == 'per_security') & (_df['country'] == 'NOR')
              & (_df['sample'] == 'oos') & (_df['validation_scheme'] == 'walk_forward')
              & _df['tau_w'].notna()].copy()
    if not len(nor):
        raise ValueError('no NOR per-security rows with tau_w')

    def _tenor_years(t):
        try:
            return float(str(t).rstrip('Yy'))
        except Exception:
            return np.nan
    nor['tenor_yrs'] = nor['tenor'].map(_tenor_years)
    piv = (nor.groupby(['tenor', 'horizon'])['tau_w'].mean().reset_index())
    tenors = sorted(piv['tenor'].unique(), key=_tenor_years)
    horizons = sorted(piv['horizon'].unique())

    fig, ax = plt.subplots(figsize=(max(7, 1.1 * len(tenors)), 5))
    # author interpretation bands for tau^w
    for lo, hi, lab in [(5, 7, 'linear-ish (5–7)'), (10, 15, 'moderate (10–15)'),
                        (20, 25, 'sharp (20–25)')]:
        ax.axhspan(lo, hi, color='grey', alpha=0.12)
        ax.text(len(tenors) - 0.4, (lo + hi) / 2, lab, va='center', ha='right',
                fontsize=8, color='dimgrey')
    width = 0.8 / max(len(horizons), 1)
    for j, h in enumerate(horizons):
        sub = piv[piv['horizon'] == h].set_index('tenor').reindex(tenors)
        xs = np.arange(len(tenors)) + (j - (len(horizons) - 1) / 2) * width
        ax.bar(xs, sub['tau_w'].values, width=width, label=f'H={h}')
    ax.set_xticks(np.arange(len(tenors)))
    ax.set_xticklabels(tenors, rotation=0)
    ax.set_xlabel('NOR tenor'); ax.set_ylabel(r'weighted smoothness $\tau^{w}$')
    ax.set_title(r'Pooled HTBoost weighted smoothness $\tau^{w}$ by NOR tenor and horizon')
    ax.legend(title='horizon', fontsize=8, ncol=2)
    _finish(fig, 'fig_tau_w_by_tenor_horizon.png',
            'Headline smoothness diagnostic: importance-weighted geometric-mean tau^w '
            '(paper eq.19, capped at 40) for each NOR tenor, per horizon. Shaded bands = '
            'author thresholds 5-7 (near-linear), 10-15 (moderate), 20-25 (sharp).')
except Exception as e:
    print(f'[fig a: tau_w] skipped — {repr(e)[:80]}')


# ── (b) Partial-dependence of the most important features of a representative NOR fit ─
try:
    if not REP_FIT.get('output'):
        raise ValueError('no representative fit retained (REP_FIT empty — run the H=%d '
                         'walk-forward cell in this kernel)' % REP_H)
    _out = REP_FIT['output']
    _xtr = REP_FIT.get('x_tr')
    if _xtr is None or not len(_xtr):
        raise ValueError('representative fit has no feature matrix to perturb')
    _imp = REP_FIT.get('imp', {}) or {}
    # rank numeric features by importance (categoricals handled as bars elsewhere)
    _num = [c for c in _xtr.columns if pd.api.types.is_numeric_dtype(_xtr[c])]
    _ranked = [f for f, _ in sorted(_imp.items(), key=lambda kv: -kv[1]) if f in _num]
    _feats = (_ranked[:4] if _ranked else _num[:4])
    if not _feats:
        raise ValueError('no numeric features available for partial dependence')

    def _pdp(feature, npts=25, sample=400):
        """Model partial dependence via the retained model's own predictions: sweep the
        feature over its observed quantile grid, holding the (sampled) rows otherwise
        fixed, and average HTBpredict. Tries the package's HTBpartialplot first."""
        base = _xtr if len(_xtr) <= sample else _xtr.sample(sample, random_state=JULIA_SEED)
        grid = np.quantile(_xtr[feature].to_numpy(float),
                           np.linspace(0.02, 0.98, npts))
        grid = np.unique(grid)
        ys = []
        for g in grid:
            b = base.copy(); b[feature] = g
            yh = np.asarray(jl.HTBpredict(jl.DataFrame(b), _out), dtype=float)
            ys.append(float(np.mean(yh)))
        return grid, np.asarray(ys)

    # Best-effort: try the package's native partial-dependence function once (guarded).
    try:
        _ = jl.HTBpartialplot
        _have_pkg_pdp = True
    except Exception:
        _have_pkg_pdp = False

    ncol = 2; nrow = int(np.ceil(len(_feats) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(11, 3.2 * nrow), squeeze=False)
    for k, feat in enumerate(_feats):
        ax = axes[k // ncol][k % ncol]
        try:
            gx, gy = _pdp(feat)
            ax.plot(gx, gy, marker='o', ms=3, lw=1.5)
            ax.set_title(feat, fontsize=9)
            ax.set_xlabel('feature value'); ax.set_ylabel('mean predicted Δ')
            ax.grid(alpha=0.3)
        except Exception as _pe:
            ax.text(0.5, 0.5, f'skipped\n{repr(_pe)[:40]}', ha='center', va='center',
                    transform=ax.transAxes, fontsize=8)
    for k in range(len(_feats), nrow * ncol):
        axes[k // ncol][k % ncol].axis('off')
    fig.suptitle(f'Partial dependence — representative pooled fit '
                 f'(H={REP_FIT.get("H")}, fold={REP_FIT.get("fold")}); '
                 f'pkg HTBpartialplot available={_have_pkg_pdp}', fontsize=11)
    _finish(fig, 'fig_partial_dependence_rep_fit.png',
            'Recovered effect of the top features on the predicted change for a single '
            'retained NOR-relevant pooled model. Smooth monotone curves => smooth effects; '
            'steps/kinks => sharp effects. Computed from the retained model via HTBpredict '
            'on a feature-swept grid (package HTBpartialplot probed for availability).')
except Exception as e:
    print(f'[fig b: partial dependence] skipped — {repr(e)[:90]}')


# ── (c) Feature importance by taxonomy bucket (NOR results), per horizon ─────
try:
    if not _imps:
        raise ValueError('no importance records available')
    recs = []
    for r in _imps:
        if not r.get('use_categoricals', True):
            continue
        b = {}
        for feat, val in r['imp'].items():
            bk = bucket_feature(feat)
            b[bk] = b.get(bk, 0.0) + float(val)
        b['horizon'] = r['horizon']
        recs.append(b)
    if not recs:
        raise ValueError('no with-categoricals importance records')
    buck = pd.DataFrame(recs).fillna(0.0).groupby('horizon').mean(numeric_only=True)
    # normalise each horizon to shares for readability
    buck_share = buck.div(buck.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    horizons = list(buck_share.index)
    cols = list(buck_share.columns)

    fig, ax = plt.subplots(figsize=(max(7, 1.4 * len(horizons)), 5))
    bottom = np.zeros(len(horizons))
    for c in cols:
        vals = buck_share[c].to_numpy(float)
        ax.bar([str(h) for h in horizons], vals, bottom=bottom, label=c)
        bottom += vals
    ax.set_xlabel('horizon (days)'); ax.set_ylabel('share of total importance')
    ax.set_title('Pooled feature importance by taxonomy bucket, per horizon (with cats)')
    ax.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.01, 1), loc='upper left')
    _finish(fig, 'fig_importance_buckets_by_horizon.png',
            'Stacked share of HTBoost feature importance by taxonomy bucket '
            '(curve/momentum/vol/carry/credit/Norway/cross-market/macro/categorical), '
            'averaged across walk-forward folds, per horizon.')
except Exception as e:
    print(f'[fig c: importance buckets] skipped — {repr(e)[:80]}')


# ── (d) OOS CT R^2 and DirAcc by horizon × regime — NOR-pool aggregate ───────
try:
    nor_agg = _df[(_df['agg_level'] == 'aggregate_nor') & (_df['sample'] == 'oos')
                  & (_df['validation_scheme'] == 'walk_forward')
                  & (_df['use_categoricals'] == True)].copy()
    if not len(nor_agg):
        raise ValueError('no NOR-pool aggregate OOS rows')
    da = nor_agg.pivot_table(index='horizon', columns='regime', values='dir_acc', aggfunc='mean')
    ct = nor_agg.pivot_table(index='horizon', columns='regime', values='ct_r2_oos', aggfunc='mean')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    for ax, mat, title, center, fmt in [
            (axes[0], da, 'Directional accuracy (chance=0.50)', 0.50, '{:.2f}'),
            (axes[1], ct, 'Campbell–Thompson OOS $R^2$ (chance=0)', 0.0, '{:+.3f}')]:
        vmax = np.nanmax(np.abs(mat.to_numpy(float) - center)) or 0.01
        im = ax.imshow(mat.to_numpy(float), cmap='RdBu_r', aspect='auto',
                       vmin=center - vmax, vmax=center + vmax)
        ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=30, ha='right')
        ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index)
        ax.set_xlabel('regime'); ax.set_ylabel('horizon (days)'); ax.set_title(title)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                v = mat.to_numpy(float)[i, j]
                if np.isfinite(v):
                    ax.text(j, i, fmt.format(v), ha='center', va='center', fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle('NOR-pool aggregate OOS by horizon × regime (walk-forward, with cats)',
                 fontsize=12)
    _finish(fig, 'fig_nor_ct_r2_diracc_heatmap.png',
            'Headline NOR-pool out-of-sample skill: directional accuracy (chance 0.50) '
            'and Campbell-Thompson R^2 (chance 0) by horizon and regime. Colour centred on '
            'the chance line; blue = better than chance, red = worse.')
except Exception as e:
    print(f'[fig d: CT R2 / DA heatmap] skipped — {repr(e)[:80]}')


print(f'\n[FIGURES] {len(_saved_figs)} figure(s) written to {FIG_DIR}/:')
for _p in _saved_figs:
    print(f'  - {_p}')

### Verdict
State the decision criteria fixed before running, then report the observed pooled walk-forward train/OOS gap, the multiple-testing survivors, the value-of-categoricals delta, and the block-CV result under its ex-ante asymmetric interpretation rule.

In [ ]:
# ── VERDICT (pooled v5) — null confirmation + block-CV asymmetric interpretation ─
print('=== pooled v5 VERDICT ===\n')
_AGGc = (df_all['agg_level'] == 'aggregate_nor') & (df_all['use_categoricals'] == True)
oos_wf = df_all[_AGGc & (df_all['validation_scheme'] == 'walk_forward') & (df_all['sample'] == 'oos')]
tr_wf  = df_all[_AGGc & (df_all['validation_scheme'] == 'walk_forward') & (df_all['sample'] == 'train')]

mean_oos   = float(oos_wf['dir_acc'].mean()) if len(oos_wf) else float('nan')
mean_train = float(tr_wf['dir_acc'].mean())  if len(tr_wf)  else float('nan')
mean_gap   = mean_train - mean_oos
mean_ctr2  = float(oos_wf['ct_r2_oos'].mean()) if len(oos_wf) else float('nan')

print('Decision criteria (stated before running):')
print('  Null confirmed: aggregate OOS DirAcc ~ noise AND no {horizon×fold} clears MTC.')
print('  Genuine lead:   clears Bonferroni/BH-FDR AND positive across MULTIPLE regimes.')
print()
print('Observed (pooled NOR, walk-forward aggregate, with categoricals):')
print(f'  mean train DirAcc:            {mean_train:.4f}')
print(f'  mean OOS   DirAcc:            {mean_oos:.4f}   (noise ≈ 0.50)')
print(f'  mean overfit gap (train−OOS): {mean_gap:+.4f}')
print(f'  mean Campbell–Thompson R²:    {mean_ctr2:+.4f}   (negative ⇒ worse than RW)')
print(f'  walk-forward MTC survivors:   Bonferroni={b_wf}  BH-FDR={h_wf}  (N={N_wf})')
if len(oos_wf):
    print('\n  Aggregate OOS DirAcc by regime:')
    for reg, m in oos_wf.groupby('regime')['dir_acc'].mean().items():
        print(f'    {reg:<8s} {m:.4f}')

# Value of categoricals one-liner (WI4).
if 'cat_delta' in dir() and len(cat_delta) and 'd_dir_acc' in cat_delta:
    print(f'\n  Value of categoricals (mean Δ OOS DirAcc, cats − no-cats): '
          f'{cat_delta["d_dir_acc"].mean():+.4f}')

print('\nVerdict:')
print('─' * 78)
if b_wf == 0 and h_wf == 0:
    print(
        f'Pooled v5 confirms the null. Across the {{horizon × fold}} grid the single pooled model\'s '
        f'aggregate walk-forward OOS DirAcc averages {mean_oos:.3f} (in-sample {mean_train:.3f}, gap '
        f'{mean_gap:+.3f}); mean Campbell–Thompson R² is {mean_ctr2:+.3f} (no improvement on the '
        f'random walk). No {{horizon×fold}} cell survives Bonferroni or BH-FDR (N={N_wf}). Pooling '
        f'every security with security/country/maturity categoricals — i.e. letting one model borrow '
        f'cross-sectional strength — does NOT beat the random walk OOS.'
    )
else:
    print(
        f'CAUTION: {b_wf} Bonferroni / {h_wf} BH-FDR survivor(s) in the walk-forward family (N={N_wf}). '
        f'Apply the NOR_30Y standard: require positive DirAcc across MULTIPLE regimes, re-audit for '
        f'leakage, and check that the value-of-categoricals delta is positive across regimes too.'
    )
print('─' * 78)

# Block-CV / LORO asymmetric interpretation (robustness, never the headline).
oos_bl = df_all[_AGGc & (df_all['sample'] == 'oos')
                & (df_all['validation_scheme'].isin(['block_cv', 'loro']))]
if len(oos_bl):
    mean_bl = float(oos_bl['dir_acc'].mean())
    print('\nBlock-CV / LORO robustness (NON-causal — learnability diagnostic, NOT a forecast):')
    print(f'  mean block-CV/LORO aggregate OOS DirAcc: {mean_bl:.4f}   '
          f'(non-causal MTC: Bonferroni={b_bl} BH-FDR={h_bl}, N={N_bl})')
    print('  Ex-ante asymmetric rule:')
    if abs(mean_bl - 0.5) < 0.02:
        print('   → ≈ 0.50: STRONGER confirmation of the null (future data transfers nothing).')
    elif mean_bl > mean_oos + 0.02:
        print('   → > walk-forward while wf ≈ 0.50: FUTURE-INFORMATION / training-size artefact;')
        print('     flags NON-STATIONARITY, NOT tradeable skill. Never enters Part C.')
    else:
        print('   → comparable to walk-forward: consistent with the null.')
    print('  (Bergmeir-Hyndman-Koo 2018; López de Prado 2018 purged K-fold + embargo.)')
print('\n[pooled v5 COMPLETE]')